# H-AMR Visualization Tutorial

Author: Christopher J. Bambic

This tutorial is broken into 8 parts. 

I) Compiling, Reading Data, and Plotting  
II) Analysis  
III) Radial Profiles and Time Series  
IV) Poloidal Slices and Stresses   
V) Vertical Profiles   
VI) Movies   
VII) Three-dimensional Rendering  
VIII) Ray-tracing with RAPTOR

The goal is to provide an overview of all of the functions available in the new analysis toolkit. As new modules are added (raytracing, binary black holes, etc.), the tutorial will be updated. For now, this notebook is designed as a standalone tutorial appropriate for a new H-AMR user. I assume the user knows general relativity and relativistic hydrodynamics/electrodynamics as well as plasma physics and radiation transport at the level required to work with H-AMR simulations.

## I) Compiling, Reading Data, and Plotting

## Compiling

The directory structure for the new H-AMR workflow is the following:

- Averages
    - Transition_XRB_HighMdot_MAD
    - Transition_XRB_HighMdot_SANE
        - Full_Resolution
            - Energy_Fluxes
            - Mass_Fluxes
            - Profiles
            - Slices
            - Stresses
            - Timescales
        - Reduced_Resolution
    - T65MAD
- DATA
    - My_Simulation
- HAMR_Visualization
    - Analyze_HAMR_Data
        - analysis.py
        - check_if_done.py
        - ...
    - Plot_HAMR_Data
        - plot_profiles.py
        - plot_slices.py
        - plot_stresses.py
        - plot_time_series.py
        - ...
    - Read_HAMR_Data
        - hamr_read_c.cpp
        - hamr_read_c.pyx
        - hamr_read.cpp
        - hamr_read.h
        - hamr_read.py
        - setup.py
    - Render_Movies
        - Midplane
        - Poloidal
            - cooling_rates_poloidal.py
            - density_poloidal.py
            - ...
        - QuickView
- Images
    - Transition_XRB_HighMdot_MAD
    - Transition_XRB_HighMdot_SANE
        - Full_Resolution
            - Midplane
            - Poloidal
                - Density_40Rg
                - ...
            - QuickView
        - Reduced_Resolution
    - T65MAD
- Logs
    - HAMR
    - SLURM
- Movies
    - Transition_XRB_HighMdot_MAD
    - Transition_XRB_HighMdot_SANE
        - Full_Resolution
          - Poloidal
            - Density_40Rg
            - ...
    - T65MAD
- PDF
    - Radial_Profiles
    - Time_Series
    - Vertical_Profiles

- RUN
    - Jupyter
        - Tutorial.ipynb
    - SLURM_Outputs

This is not a requirement; however, it will help with organization for all of the data products we'll be working with. For instance, I'll be assuming that you're in the `RUN/Jupyter` directory when executing this tutorial notebook. If you are not, you may run into path issues. For now, as long as you have ``RUN`` and ``HAMR_Visualization``, you should be fine. As each user will have different desires for their layout, I note that all of the paths for the toolkit are defined in `HAMR_Visualization/Analyze_HAMR_Data/compute_variables.py`. There, you have access to the following paths:

asys_path   = str(this_file.parent)  
vis_path    = str(this_file.parent.parent)  
user_path   = str(this_file.parent.parent.parent)  
shared_path = str(this_file.parent.parent.parent.parent)  # Modify shared path  

Generally, you should only need to adjust the `shared_path` variable. The ``shared_path`` is where data shared by the entire collaboration is hosted. For any scripts in the toolkit, you need to append the ``--shared`` flag to a script to access data from the shared simulations. On the Andes cluster, this directory is ``/lustre/orion/ast178/proj-shared``. Currently, I am assuming that the ``Averages``, ``HAMR_Visualization``, and ``RUN`` directories are all under ``/lustre/orion/ast178/proj-shared/cbambic`` as by username is cbambic.

To compile the toolkit execute the command:

In [ ]:
%run ../../HAMR_Visualization/Read_HAMR_Data/setup.py build_ext --inplace

There may be issues with compiling (usually related to missing libraries/dependencies). The toolkit uses OpenMP for reading in data and requires C++17 or greater. Also, check that your NumPy versions are up-to-date. Assuming this works, you should now have an executable in your ``RUN`` directory that looks something like this, ``hamr_readc.cpython-310-x86_64-linux-gnu.so``. To test that you have compiled the toolkit properly, try running the next block of code.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

You will likely need this code block at the top of any Jupyter notebook you create.

We can now read in the data!

H-AMR simulation data is distributed across 4 different files: 

1. grid file (``Simulation_Directory/gdumps/grid``)  
2. parameters file (``Simulation_Directory/dumps{file_number}/parameters``)  
3. data dumps (``Simulation_Directory/dumps{file_number}/new_dump{GPU_ID}``)  
4. grid dumps (``Simulation_Directory/gdumps/gdump{Global_Block_ID}``)

Here, ``file_number`` refers to the dump number, ``GPU_ID`` refers to the unique ID for the GPU that wrote the ``new_dump`` file (for a 16-GPU run, the ID is in the range 0-15), and the ``Global_Block_ID`` is an index in the range of all possible AMR blocks. The simulation domain is divided into blocks that are a set number of zones in size. We use the variables ``bs1``, ``bs2``, and ``bs3`` to denote these sizes in the x1, x2, and x3 coordinates, respectively. A block consists of bs1 x bs2 x bs3 zones. Since H-AMR is an adaptive mesh refinement (AMR) code that utilizes octree refinement, the domain can change with each timestep and blocks can created and destroyed. There exists a "global block array" that identifies all possible blocks that can be created from the grid via refinement of the blocks in the "base" grid. The gdumps are numbered based on the ID in this "global block array." So, suppose that the base grid consists of a single block but 5 levels of AMR are allowed. There are a total of 8^5 = 32768 blocks that can be created. The gdumps thus take on a value between 0 and 32767. As you can see, the number of gdumps files can grow rapidly. We only write out files corresponding to blocks that are actually created during the code's execution. For instance, in our example with 5 levels of AMR, if we never activated refinement, there would be a single gdump written, ``gdumps/gdump0``. 

## Reading in H-AMR Data

The grid file contains the "block" array, which is an object that contains all of the information required for the AMR grid structure. You as the user should never have to interact with the block array. The grid file is read with initialization of the H-AMR Read class. Try executing the following code.

In [ ]:
simulation_dir = f'{cv.shared_path}/TRANS_XRB_HIGH_SANE'
H = hr.Read(simulation_dir)
# List all attributes of the class
print(dir(H)) 

Throughout this tutorial, "H" refers to an object of the H-AMR Read class. You can see that the H object contains the "block" array after initialization. You can also see all of the options available to you. 

Let's go through them.

Read class options (alphabetical, mostly): 
- ``axisym`` (default: True): use an axisymmetric metric.
- ``by_block`` (default: False): read in the data "by block" instead of upscaling the data to a global uniform grid.
- ``check_files`` (default: False): check that the appropriate grid dumps exist.
- ``cgs_data`` (default: False): apply cgs units to the primitive variables (B has units of Gauss, uu has units of c, Te has units of temperature, etc.).
- ``cgs_grid`` (default: False): apply cgs units to the grid variables (r has units of cm instead of gravitational radii $R_g$).
- ``coords1D`` (default: False): load in grid coordinates as 1D arrays instead of 3D arrays to save memory.
- ``lowres1`` (default: 1): downsampling factor in the x1 direction.
- ``lowres2`` (default: 1): downsampling factor in the x2 direction.
- ``lowres3`` (default: 1): downsampling factor in the x3 direction.
- ``MBH`` (default: 1): mass of the black hole in units of $M_{\odot}$ (required if cgs_grid is True).
- ``resolution`` (default: False): add the resolution level ``level_x1`` to the grid dictionary.
- ``r_min`` (default: H.params['Rin']): minimum radius to be read in.
- ``r_max`` (default: H.params['Rout']): maximum radius to be read in.
- ``theta_min`` (default: ($\pi$/2) * (H.params['startx2'] + 1)): minimum theta ($\theta$) value to be read in.
- ``theta_max`` (default: $\pi$): maximum theta ($\theta$) value to be read in.
- ``phi_min`` (default: H.params['startx3']): minimum phi ($\varphi$) value to be read in.
- ``phi_max`` (default: $2 \pi$): maximum phi ($\varphi$) value to be read in.
- ``start_block`` (default: 0): starting block ID (local block ID) to be read in.
- ``end_block`` (default: nb): ending block ID (local block ID) to be read in.
- ``i_min`` (default: 0): starting index in the x1 direction to be read in.
- ``i_max`` (default: nx): starting index in the x1 direction to be read in.
- ``j_min`` (default: 0): starting index in the x2 direction to be read in.
- ``j_max`` (default: ny): starting index in the x2 direction to be read in.
- ``k_min`` (default: 0): starting index in the x3 direction to be read in.
- ``k_max`` (default: nz): starting index in the x3 direction to be read in.

The Kerr metric is axisymmetric, so we default to an axisymmetric grid. Common practice in the H-AMR Collaboration is to read all data in as a "unigrid" or a uniform spherical grid. This has the advantage of being intuitively straightforward to work with. The downside is that for many levels of mesh refinement, we must upscale regions at low levels of refinement to the highest refinement level. So, suppose a grid consists of a two blocks in the base grid and 1 level of refinement is enabled. We refer to these blocks as Blocks A and B. If Block A is refined and Block B is not refined, then Block A breaks into 8 additional blocks (Blocks A0, A1, ... A7). There are now 9 total blocks in the simulation. If the ``by_block`` flag is not enabled, the data in block B will be doubled in each direction to get Block B to the same refinement level as Block A. So, if the blocks have size bs1 x bs2 x bs3, the real simulation is performed with 9 blocks of size bs1 x bs2 x bs3. However, the unigrid array will have size 8 sub-blocks x 2 base-blocks x bs1 x bs2 x bs3 or 16 x bs1 x bs2 x bs3 total zones. 

For any analysis tasks prior to "production" scale, the user should implement at least one of the following strategies:
1. Downsample data in the 1, 2, or 3 directions using the ``lowres1`` etc. parameters.
2. Select data based on a range in $r \in$ (``r_min``, ``r_max``), $\theta \in$ (``theta_min``, ``theta_max``), $\varphi \in$ (``phi_min``, ``phi_max``).
3. Select data based on index ranges of $i \in$ (``i_min``, ``i_max``) etc.
4. Read in data ``by_block``.

### Parameters Dictionary

The simulation's parameters are contained in the parameters dictionary (``H.params``). Any time you read in the grid via ``H.read_grid(dump_number)`` or the data ``H.read_prims(dump_number)``, you will load in the parameters dictionary. You can also load in the parameters dictionary separately. Execute the following to load in the parameters for dump 9000:

In [ ]:
H.read_params(9000)
# List keys in params dictionary.
print(H.params.keys())

These parameters are defined as follows (see also the end of ``HAMR_Visualization/Read_HAMR_Data/hamr_read.py``:

- t: current time in Rg/c
- n_active: number of blocks on node 0
- n_active_total: total number of active blocks ("active" means currently operating for that dump)
- nstep: current step of output (dump) file
- Dtd: time between outputs (dumps)
- Dtr: time between restart dumps
- dump_cnt: current number of output/dump file (fnum)
- rdump_cnt: number of most recent restart file for output fnum 
- dt: timestep in Rg/c
- failed: 0 or 1 (1 means output file did not write properly)
- bs1, bs2, bs3: block sizes in each dimension
- nb1, nb2, nb3:  number of blocks at the least refined AMR level
- nmax: total number of possible blocks
- startx1, startx2, startx3:  start coordinates in each dimension
- dx1, dx2, dx3: grid spacing in internal coordinates at 0th refinement level
- tf: final (stop) time
- a: dimensionless black hole spin
- gam: adiabatic index (ratio of specific heats)
- cour: CFL number
- Rin: inner radius in Rg
- Rout: outer radius in Rg
- R0: meaningless (removed from parameter dictionary)
- density scale: $\rho_0$ in CGS units

### Grid Dictionary

To read in the grid for dump 9000, execute:

In [ ]:
H.read_grid(9000, direct=True)
# List all of the keys in the grid dictionary.
print(H.grid.keys())

The ``read_grid`` function often takes the following arguments:
- direct (default: False): Calculate the grid analytically (directly) rather than reading in the grid variables from the gdumps files.
- skip_read_params (default: False): If you already read in the parameters file, ``read_grid`` will read it again unless you change this flag to True.

The following grid variables should be contained in the grid dictionary:

- x1, x2, x3: internal coordinate arrays ($x_1 = \log{\left(r / R_g \right)}$, $x_2 = (2/\pi) \theta - 1$, $x_3 = \varphi$) for modified spherical Kerr-Schild coordinates.
- r, theta, phi: spherical coordinate arrays.
- gcov: metric tensor ($g_{\mu \nu}$).
- gcon: inverse of the metric tensor ($g^{\mu \nu}$).
- gdet: metric determinant $\sqrt{-g} \equiv \sqrt{-\rm{det}(g_{\mu \nu})}$.
- dxdxp: transformation matrix from internal to spherical Kerr-Schild coordinates. 

Any 3D H-AMR variables are actually 4D arrays where the leading index is the "block index." For a uniform grid, the block index is always 0. If the data is read ``by_block``, then the block index refers to the local block ID. Let's print out the shapes of the H-AMR data arrays.

In [ ]:
for grid_key in H.grid.keys():
    print(grid_key, H.grid[grid_key].shape)

As you can see, the x1, x2, x3, r, theta, and phi arrays are all 4D arrays organized by (block ID, i, j, k), where i, j, k refer to the x1, x2, and x3 (r, theta, and phi) indices, respectively. The metric, inverse metric, and coordinate transformation matrix are all rank 2 tensors with indices that run over $\mu \in (0, 1, 2, 3)$, so they are 6D arrays, with the leading two slots corresponding to $\mu$ and $\nu$ in, e.g., $g_{\mu \nu}$. The metric determinant is a 4D array as well. Note that because the metric is axisymmetric (``axisym = True``), the metric, inverse metric, and transformation matrix have only one element in the last slot, i.e. the k or phi index.

If we were to use ``coords1D = True``, the shapes of these arrays would change.

In [ ]:
simulation_dir = f'{cv.shared_path}/TRANS_XRB_HIGH_SANE'
read_args = {
    'coords1D' : True
}
H = hr.Read(simulation_dir, **read_args)
H.read_grid(9000, direct=True)

for grid_key in H.grid.keys():
    print(grid_key, H.grid[grid_key].shape)

### Data Dictionary

To read in the primitive variables for dump 9000, execute:

In [ ]:
H.read_prims(9000)
# List the primitive variables and their shapes.
for data_key in H.data.keys():
    print(data_key, H.data[data_key].shape)

We can also load in the data ``by_block``:

In [ ]:
simulation_dir = f'{cv.shared_path}/TRANS_XRB_HIGH_SANE'
read_args = {
    'by_block' : True
}
H = hr.Read(simulation_dir, **read_args)
H.read_prims(9000)
# List the primitive variables and their shapes.
for data_key in H.data.keys():
    print(data_key, H.data[data_key].shape)

Note how the total size of the arrays hasn't changed; however, the block index now takes values other than 0. 

The primitive variables are the following (code units / cgs units):
- rho: fluid-frame rest mass density ($\rho_0$ / g/cm^3).
- u_gas: fluid-frame gas internal energy ($\rho_0 c^2$ / erg/cm^3).
- uu: 4-velocity as measured by coordinate observer ($c$ / cm/s).
- B: 3-magnetic field as measured by coordinate observer ($\sqrt{4 \pi \rho_0} c$ / Gauss (G)).
- u_rad: fluid-frame radiation energy density ($\rho_0 c^2$ / erg/cm^3).
- uu_rad: 4-velocity of frame in which radiation distribution is isotropic in frame of coordinate observer ($c$ / cm/s).
- Te: temperature of electrons ($T_{e, \rm{code}} = k_B T_e / \mu_e m_p c^2$ / K).
- Ti: temperature of ions ($T_{i, \rm{code}} = k_B T_i / \mu_i m_p c^2$ / K).

Let's try reading in the data as a unigrid, but downsampling in the x1, x2, and x3 directions. We'll also apply cgs units.

In [ ]:
simulation_dir = f'{cv.shared_path}/TRANS_XRB_HIGH_SANE'
read_args = {
    'by_block' : False,
    'cgs_data' : True,
    'lowres1'  : 2,
    'lowres2'  : 2,
    'lowres3'  : 2,
}
H = hr.Read(simulation_dir, **read_args)
H.read_prims(9000)

# List the primitive variables and their shapes.
for data_key in H.data.keys():
    print(data_key, H.data[data_key].shape)

This simulation contains all of the variables for a two-temperature radiation simulation. We can check that this is the physics operating in the simulation using the ``physics`` dictionary, which is generally not to be modified by the user. However, we can access the ``physics`` dictionary from the Read object.

In [ ]:
for physic in H.physics.keys():
    print(physic, f"Active? : {bool(H.physics[physic])}")

As you can see, the following physics modules are available:

- RAD_M1: Radiation transport using gray opacities and the M1 closure approximation.
- RESISTIVE: Co-evolution of the non-ideal resistive electric field for use in reconnection studies. Uses a causal prescription for the resistive electric field.
- TWO_T: Co-evolution of the ion and electron entropy equations via a tracer fluid to evolve the ion and electron temperatures subject to differential heating of the species and Coulomb collisional coupling between ions and electrons.
- P_NUM: Photon number conserving Comptonization.
- DO_YE: Track the electron fraction.
- NEUTRINOS_M1: Neutrino transport using energy-averaged cross-sections and the M1 closure approximation.
- DO_NUCLEAR: Track nuclear abundances for the purpose of evolving nuclear rate equations.
- DOTRACER: Coevolve passive tracer fluids (useful for tracking floored material).
- DUMP_DISS: Dump the dissipation diagnostic when the entropy equations are coevolved in the simulation.

Note that if we were to load in multiple simulations, we might have different physics at work. Let's look at an example. We're going to load in data from the shared simulations ``T65MAD``, ``TRANS_XRB_HIGH_SANE``, and ``TRANS_XRB_HIGH_MAD``, and look at what physics modules are active in each and what variables appear.

In [ ]:
sims = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
file_numbers = [1500, 9000, 9000]
H = dict.fromkeys(sims)
read_args = {
    'lowres1' : 4,
    'lowres2' : 4,
    'lowres3' : 4
}
for sim, file_number in zip(sims, file_numbers):
    H[sim] = hr.Read(f'{cv.shared_path}/{sim}', **read_args)
    H[sim].read_params(file_number)
    H[sim].read_prims(file_number, skip_read_params=True)
    print(f'{sim}:')
    for physic in H[sim].physics.keys():
        if (bool(H[sim].physics[physic])):
            print(physic, f"Active? : {bool(H[sim].physics[physic])}")
    print(f'Primitive variables: {H[sim].data.keys()}')

Because the T65MAD simulation was not evolved with radiation transport and two-temperature effects only the density, gas internal energy, 4-velocity, and 3-magnetic field are included as primitive variables. Now that we can load in the data, it's time to make some initial figures.

Before continuing, please go to "Edit: Clear Outputs of All Cells." This will help prevent the kernel from crashing.

## Exploring H-AMR Data

When looking at a new H-AMR simulation, the first task is to determine how many data files are available and what resolutions are used for those data files. This exploration of the data is accomplished by the ``HAMR_Visualization/Analyze_HAMR_Data/get_resolution.py`` script. With (nearly) all scripts in the toolkit, if you execute the code with the ``-h`` flag applied, usage instructions will appear on the screen.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/get_resolution.py -h

We now run this on our 3 simulations. You may want to get a coffee at this phase since this might take a moment.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/get_resolution.py T65MAD --shared
%run ../../HAMR_Visualization/Analyze_HAMR_Data/get_resolution.py TRANS_XRB_HIGH_SANE --shared
%run ../../HAMR_Visualization/Analyze_HAMR_Data/get_resolution.py TRANS_XRB_HIGH_MAD --shared

## Plotting H-AMR Data

The standard plotting routines in the H-AMR Visualization toolkit are contained in ``HAMR_Visualization/Analyze_HAMR_Data/figs.py``. Often we want to make slices of the midplane (so-called meridional or "XY" slices) or slices at a fixed value of phi (poloidal, "XZ", or "RZ" slices). 

Most of the plotting routines take arguments X, Y, Q (or for poloidal plots, X, Z, Q or R, Z, Q, etc.), where X, Y, and Q are dictionaries. For midplane slices, the dictionaries need to have the key 'r' for "right" and for poloidal slices, they need to have the keys 'l' and 'r' for "left" and "right," respectively. The reason is that I think of the poloidal plots as being two separate figures: a left side and a right side. Most errors I have seen with others using the plotting routines come from not understanding that the coordinate arrays are dictionaries and that the 'r' and 'l' keys need to be in said dictionaries. 

Let's look at some examples. First, we'll make midplane slices for our three simulations.

### Scalar Plots

In [ ]:
# This allows scalar plots to be a standalone section
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# Using the torch colormap, create a log-scaled scalar plot of a scalar variable with limits 1.e-3 to 1.e3.
# FS is "fontsize" (this is a remnant of variable names during my Ph.D.).
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True
}

# For figure labels (again, remnant of names during my Ph.D.).
opt = dict(ha='left', va='center', color='white',
           bbox=dict(facecolor='none', edgecolor='none'))

sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]
H = dict.fromkeys(sims)
read_args = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'lowres1'  : 1,
    'lowres2'  : 1,
    'lowres3'  : 1,
    'r_max'    : 80. # maximum radius
}
for sim, file_number in zip(sims, file_numbers):
    H[sim] = hr.Read(f'{cv.shared_path}/{sim}', **read_args)
    H[sim].read_grid(file_number, direct=True)
    H[sim].read_prims(file_number, skip_read_params=True)

# Create dictionary for plotting axes.
axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used
for sim in sims:
    x3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.cos(H[sim].grid['phi'])
    y3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.sin(H[sim].grid['phi'])
    nth = H[sim].grid['theta'].shape[2]
    X   = {'r' : 0.5 * (x3D[0, :, nth//2 - 1, :] + x3D[0, :, nth//2, :])}
    Y   = {'r' : 0.5 * (y3D[0, :, nth//2 - 1, :] + y3D[0, :, nth//2, :])}
    rho = {'r' : 0.5 * (H[sim].data['rho'][0, :, nth//2 - 1, :] + H[sim].data['rho'][0, :, nth//2, :])}
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Great! Your first figure with H-AMR data is pretty nice already, but we can improve it. Let's look at some issues. First, there's this giant missing sliver in the figures that's a result of the simulation domain not extending strictly from 0 to $2 \pi$ (it stops just short of $2 \pi$). We can fix this issue with the "pole" of the simulation using the ``fix_pole`` keyword, which takes values of "average" (averaging the points closest to 0 and $2 \pi$) and "constant" (copying over the points closest to 0 and $2 \pi$ to fill in the sliver).  

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim in sims:
    x3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.cos(H[sim].grid['phi'])
    y3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.sin(H[sim].grid['phi'])
    nth = H[sim].grid['theta'].shape[2]
    X   = {'r' : 0.5 * (x3D[0, :, nth//2 - 1, :] + x3D[0, :, nth//2, :])}
    Y   = {'r' : 0.5 * (y3D[0, :, nth//2 - 1, :] + y3D[0, :, nth//2, :])}
    rho = {'r' : 0.5 * (H[sim].data['rho'][0, :, nth//2 - 1, :] + H[sim].data['rho'][0, :, nth//2, :])}
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

If I wrote the notebook correctly, this should throw an error. The reason is that you need to provide an r, theta, and phi array to use the ``fix_pole`` keyword argument. We'll fix that now.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim in sims:
    x3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.cos(H[sim].grid['phi'])
    y3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.sin(H[sim].grid['phi'])
    nth = H[sim].grid['theta'].shape[2]
    
    X   = {'r' : 0.5 * (x3D[0, :, nth//2 - 1, :] + x3D[0, :, nth//2, :])}
    Y   = {'r' : 0.5 * (y3D[0, :, nth//2 - 1, :] + y3D[0, :, nth//2, :])}
    rho = {'r' : 0.5 * (H[sim].data['rho'][0, :, nth//2 - 1, :] + H[sim].data['rho'][0, :, nth//2, :])}
    
    plt_args['r']     = H[sim].grid['r']
    plt_args['theta'] = H[sim].grid['theta']
    plt_args['phi']   = H[sim].grid['phi']
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Now the sliver near the pole is gone. Let's also see if we can extend the colorbars. Passing the standard "extend" keyword should do the trick here. It will take values of "min," "max," and "both." To check if the colorbar should be extended, we use the function ``compute_variables.check_extend()``. We'll also change the limits to demonstrate.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim in sims:
    x3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.cos(H[sim].grid['phi'])
    y3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.sin(H[sim].grid['phi'])
    nth = H[sim].grid['theta'].shape[2]
    
    X   = {'r' : 0.5 * (x3D[0, :, nth//2 - 1, :] + x3D[0, :, nth//2, :])}
    Y   = {'r' : 0.5 * (y3D[0, :, nth//2 - 1, :] + y3D[0, :, nth//2, :])}
    rho = {'r' : 0.5 * (H[sim].data['rho'][0, :, nth//2 - 1, :] + H[sim].data['rho'][0, :, nth//2, :])}
    
    plt_args['r']     = H[sim].grid['r']
    plt_args['theta'] = H[sim].grid['theta']
    plt_args['phi']   = H[sim].grid['phi']

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Now, in the interest of brevity, I'd like to not have to type so much to make these figures. Because midplane plots are used so much by the H-AMR Collaboration, we have a class designed to handle plotting midplane variables called ``compute_variables.Midplane_Slice``. We pass arguments to ``Midplane_Slice`` as if it's a version of the ``Read`` class, but we have to provide a dump number.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for (sim, file_number) in zip(sims, file_numbers):
    MS   = cv.Midplane_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args)
    X, Y = MS.calc_XY()
    rho  = {'r' : MS.calc_rho()}

    plt_args['r']     = MS.r
    plt_args['theta'] = MS.th
    plt_args['phi']   = MS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

This is dramatically simpler. The ``Midplane_Slice`` class will then allow us to visualize other quantities of interest.

### Streamline Plots

Let's visualize the flow around this black hole. To do so, we'll create a streamline/vector (vec) plot.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

# Create white vectors with density (dens) parameter (4.), linewidth 0.6, etc.
# See documentation for "streamplot" as this is what is called under the hood.
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for (sim, file_number) in zip(sims, file_numbers):
    MS   = cv.Midplane_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args)
    X, Y = MS.calc_XY()
    rho  = {'r' : MS.calc_rho()}

    plt_args['r']     = MS.r
    plt_args['theta'] = MS.th
    plt_args['phi']   = MS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    # Choose uniform grid to interpolate points to before drawing streamlines.
    X_uni, Y_uni = MS.calc_XY_uni(40., 100)
    uux, uuy = MS.calc_uu_cart(lr=True) # Creates the 'l' and 'r' dictionaries for you.
    Cart.plot_xy_vec(fig, axes[sim], X, Y, X_uni, Y_uni, uux, uuy, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Placing streamlines for the magnetic field on top of these is also straightforward. 

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

# Create white vectors with density (dens) parameter (4.), linewidth 0.6, etc.
# See documentation for "streamplot" as this is what is called under the hood.
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for (sim, file_number) in zip(sims, file_numbers):
    MS   = cv.Midplane_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args)
    X, Y = MS.calc_XY()
    rho  = {'r' : MS.calc_rho()}

    plt_args['r']     = MS.r
    plt_args['theta'] = MS.th
    plt_args['phi']   = MS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    # Choose uniform grid to interpolate points to before drawing streamlines.
    vec_args['color'] = 'white'
    X_uni, Y_uni = MS.calc_XY_uni(40., 100)
    uux, uuy = MS.calc_uu_cart(lr=True) # Creates the 'l' and 'r' dictionaries for you.
    Cart.plot_xy_vec(fig, axes[sim], X, Y, X_uni, Y_uni, uux, uuy, **vec_args)

    vec_args['color'] = 'cyan'
    X_uni, Y_uni = MS.calc_XY_uni(40., 100)
    Bx, By = MS.calc_B_cart(lr=True) # Creates the 'l' and 'r' dictionaries for you.
    Cart.plot_xy_vec(fig, axes[sim], X, Y, X_uni, Y_uni, Bx, By, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Admittedly, this is pretty chaotic. One way we could clean this up is by masking out fieldlines where they're not of interest. Let's focus on the magnetic fields in the inner 20 gravitational radii.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

# Create white vectors with density (dens) parameter (4.), linewidth 0.6, etc.
# See documentation for "streamplot" as this is what is called under the hood.
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for (sim, file_number) in zip(sims, file_numbers):
    MS   = cv.Midplane_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args)
    X, Y = MS.calc_XY()
    rho  = {'r' : MS.calc_rho()}

    plt_args['r']     = MS.r
    plt_args['theta'] = MS.th
    plt_args['phi']   = MS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    # Choose uniform grid to interpolate points to before drawing streamlines.
    vec_args['color'] = 'white'
    X_uni, Y_uni = MS.calc_XY_uni(40., 100)
    Bx, By = MS.calc_B_cart(lr=True) # Creates the 'l' and 'r' dictionaries for you.
    vec_args['mask'] = {'r' : np.sqrt(X['r'] * X['r'] + Y['r'] * Y['r']) > 20.}
    Cart.plot_xy_vec(fig, axes[sim], X, Y, X_uni, Y_uni, Bx, By, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Finally, a fairly recent development in Matplotlib was the ``broken_streamlines = False`` option. A number of authors have found that they can get some pretty visualizations using that setting.

In [ ]:
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'average'
}

# Create white vectors with density (dens) parameter (4.), linewidth 0.6, etc.
# See documentation for "streamplot" as this is what is called under the hood.
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : False,
   'arrowsize'          : 1.0
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for (sim, file_number) in zip(sims, file_numbers):
    MS   = cv.Midplane_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args)
    X, Y = MS.calc_XY()
    rho  = {'r' : MS.calc_rho()}

    plt_args['r']     = MS.r
    plt_args['theta'] = MS.th
    plt_args['phi']   = MS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xy(fig, axes[sim], X, Y, rho, **plt_args)

    # Choose uniform grid to interpolate points to before drawing streamlines.
    vec_args['color'] = 'white'
    X_uni, Y_uni = MS.calc_XY_uni(40., 100)
    Bx, By = MS.calc_B_cart(lr=True) # Creates the 'l' and 'r' dictionaries for you.
    vec_args['mask'] = {'r' : np.sqrt(X['r'] * X['r'] + Y['r'] * Y['r']) > 20.}
    Cart.plot_xy_vec(fig, axes[sim], X, Y, X_uni, Y_uni, Bx, By, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$y \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Before continuing, please go to "Edit: Clear Outputs of All Cells." This will help prevent the kernel from crashing.

### Poloidal Plots: XZ, RZ, and Mirroring

To create poloidal plots, we'll first show the naive way and then use the ``compute_variables.Phi_Slice`` class. All of the methods in ``Phi_Slice`` are inherited by ``Midplane_Slice``; however, methods in ``Phi_Slice``often take the "phi index" as an input. This allows a single object of the ``Phi_Slice`` class to handle many poloidal slices. Let's look at some examples. First, we'll plot poloidal slices of the density by reading in the full 3D data for each data set.

In [ ]:
# This allows poloidal plots to be a standalone section
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# For figure labels (again, remnant of names during my Ph.D.).
opt = dict(ha='left', va='center', color='white',
           bbox=dict(facecolor='none', edgecolor='none'))

sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]
H = dict.fromkeys(sims)
read_args = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'lowres1'  : 1,
    'lowres2'  : 1,
    'lowres3'  : 1,
    'r_max'    : 80. # maximum radius
}
for sim, file_number in zip(sims, file_numbers):
    H[sim] = hr.Read(f'{cv.shared_path}/{sim}', **read_args)
    H[sim].read_grid(file_number, direct=True)
    H[sim].read_prims(file_number, skip_read_params=True)

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    #'fix_pole' : 'constant' # omit to demonstrate effect
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim in sims:
    x3D = H[sim].grid['r'] * np.sin(H[sim].grid['theta']) * np.cos(H[sim].grid['phi'])
    z3D = H[sim].grid['r'] * np.cos(H[sim].grid['theta']) * np.ones_like(H[sim].grid['phi'])
    nphi= H[sim].grid['phi'].shape[3]

    # Define left ('l') and right ('r') sides of the plot.
    X   = {
        'r' : x3D[0, :, :, 0],
        'l' : x3D[0, :, :, nphi//2],
    }
    Z   = {
        'r' : z3D[0, :, :, 0],
        'l' : z3D[0, :, :, nphi//2],
    }
    rho = {
        'r' : H[sim].data['rho'][0, :, :, 0],
        'l' : H[sim].data['rho'][0, :, :, nphi//2],
    }

    # Leave off to demonstrate the missing pole. 
    #plt_args['r']     = H[sim].grid['r']
    #plt_args['theta'] = H[sim].grid['theta']
    #plt_args['phi']   = H[sim].grid['phi']

    #plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Great! Now, let's make the same figure with the ``Phi_Slice`` class. We'll also fix the poles by using ``fix_pole='constant'``, and do the same extensions of the colorbar. To use the ``Phi_Slice`` class, we're actually going to create two objects of the class, one to handle the left side of the data and the other to handle the right side. 

In [ ]:
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'k_min'    : 0, # k_min = 0 means only take the first index in the phi direction
    'k_max'    : 1  # k_max = 1 such that only a single index of phi is read in
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_XZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Now, let's create the same figures with magnetic fields drawn on them.

In [ ]:
# Include vector arguments
vec_args = {
   'dens'               : 2.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_XZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    # -40 to +40 100 points in each direction
    X_uni, Z_uni = PS_r.calc_XZ_uni(40., 100)
    
    # Return both a left and right side.
    Bx, Bz           = PS_r.calc_B_cyl(0, lr=True)
    # Override the left side.
    Bx['l'], Bz['l'] = PS_l.calc_B_cyl(0, lr=False)
    # Flip the direction to work with the XZ plot
    Bx['l']         *= -1.
    
    Cart.plot_xz_vec(fig, axes[sim], X, Z, X_uni, Z_uni, Bx, Bz, **vec_args)
    
    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Ah. We have magnetic field lines, but something odd is happening at the pole. Because we are stitching two sides together, if the array slicing is not done perfectly, there's a possibility that field lines will not be continuous across the pole. Also, for mirrored plots, we do not want the field lines to cross the pole. However, using the ``cross_pole=True`` flag passed to ``vec_args`` (and to the ``plot_xz_vec`` function, we can allow field lines to be drawn through the pole.

In [ ]:
# Include vector arguments
vec_args = {
   'dens'               : 2.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : True # Allow streamlines to cross the pole
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_XZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    # -40 to +40 100 points in each direction
    X_uni, Z_uni = PS_r.calc_XZ_uni(40., 100)
    
    # Return both a left and right side.
    Bx, Bz           = PS_r.calc_B_cyl(0, lr=True)
    # Override the left side.
    Bx['l'], Bz['l'] = PS_l.calc_B_cyl(0, lr=False)
    # Flip the direction to work with the XZ plot
    Bx['l']         *= -1.
    
    Cart.plot_xz_vec(fig, axes[sim], X, Z, X_uni, Z_uni, Bx, Bz, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

While XZ plots are useful, the have a fatal flaw, namely that they can only appropriately visualize a single cut through the poloidal plane. Let's see this limitation in action. Suppose we instead wanted to cut the data at $\varphi = \pi/4$. What would happen?

In [ ]:
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'phi_min'  : np.pi/4.,
    'phi_max'  : np.pi
}

read_args_l = read_args_r.copy()

# Include vector arguments
vec_args = {
   'dens'               : 2.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : True # Allow streamlines to cross the pole
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_XZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    # -40 to +40 100 points in each direction
    X_uni, Z_uni = PS_r.calc_XZ_uni(40., 100)
    
    # Return both a left and right side.
    Bx, Bz           = PS_r.calc_B_cyl(0, lr=True)
    # Override the left side.
    Bx['l'], Bz['l'] = PS_l.calc_B_cyl(0, lr=False)
    # Flip the direction to work with the XZ plot
    Bx['l']         *= -1.
    
    Cart.plot_xz_vec(fig, axes[sim], X, Z, X_uni, Z_uni, Bx, Bz, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Uh oh. If I designed this notebook correctly, you should be seeing a colorful line down the middle of your figures. The reason is that you're looking in the XZ plane at a projection of data rotated out of said plane. In this case, a better approach would be to use cylindrical coordinates. Let's see that in action.

In [ ]:
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'phi_min'  : np.pi/4.,
    'phi_max'  : np.pi
}

read_args_l = read_args_r.copy()

# Include vector arguments
vec_args = {
   'dens'               : 2.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : True # Allow streamlines to cross the pole
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    R, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Get the cylindrical radial coordinate for the left side.
    Rl, _ = PS_l.calc_RZ(0)
    R['l'] = Rl['l']
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_xz(fig, axes[sim], R, Z, rho, **plt_args)

    # -40 to +40 100 points in each direction
    X_uni, Z_uni = PS_r.calc_XZ_uni(40., 100)
    
    # Return both a left and right side.
    Bx, Bz           = PS_r.calc_B_cyl(0, lr=True)
    # Override the left side.
    Bx['l'], Bz['l'] = PS_l.calc_B_cyl(0, lr=False)
    # Flip the direction to work with the XZ plot
    Bx['l']         *= -1.
    
    Cart.plot_xz_vec(fig, axes[sim], R, Z, X_uni, Z_uni, Bx, Bz, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$R \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Great! We appear to have solved the problem. Instead of looking at cuts through the full domain, we could just as easily look at only one half of the domain, utilizing the ``plot_RZ`` functions.

In [ ]:
# Return to original k_min = 0 and k_max = 1
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

# Include vector arguments
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : False # Do not allow streamlines to cross the pole
}

axes = dict.fromkeys(sims)
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    R, Z = PS_r.calc_RZ(0)
    rho = {
        'r' : PS_r.calc_rho(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])
    
    Cart.plot_Rz(fig, axes[sim], R, Z, rho, **plt_args)

    # 0 to +40 100 points in each direction
    R_uni, Z_uni = PS_r.calc_RZ_uni(40., 100)
    
    # Return both a left and right side.
    BR, Bz           = PS_r.calc_B_cyl(0, lr=True)
    
    Cart.plot_Rz_vec(fig, axes[sim], R, Z, R_uni, Z_uni, BR, Bz, **vec_args)

    axes[sim].set_xlim([0, 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$R \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Often, these views may be even more useful than the XZ cuts. Now, to see the true power of the ``Phi_Slice`` class, we can rotate through the data. This next section is memory intensive, so please clear the outputs first.

In [ ]:
# This allows poloidal plots to be a standalone section
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# For figure labels (again, remnant of names during my Ph.D.).
opt = dict(ha='left', va='center', color='white',
           bbox=dict(facecolor='none', edgecolor='none'))

sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]

# Load in the full 3D domain
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

# Include vector arguments
vec_args = {
   'dens'               : 4.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : False # Do not allow streamlines to cross the pole
}

axes = []
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i in range(3):
    axes.append(fig.add_subplot(gs[0, i]))

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

PS   = cv.Phi_Slice(f'{cv.shared_path}/T65MAD', file_numbers[0], **read_args_r)
nphi = PS.nphi
plt_indices = [0, nphi//4, nphi//2]

for idx, ax in zip(plt_indices, axes):
    R, Z = PS.calc_RZ(idx)
    rho = {
        'r' : PS.calc_rho(idx)
    }
    plt_args['r']     = PS.r
    plt_args['theta'] = PS.th
    plt_args['phi']   = PS.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim'])

    Cart.plot_Rz(fig, ax, R, Z, rho, **plt_args)
    
    # 0 to +40 100 points in each direction
    R_uni, Z_uni = PS.calc_RZ_uni(40., 100)
    
    # Return both a left and right side.
    BR, Bz = PS.calc_B_cyl(0, lr=True)
    
    Cart.plot_Rz_vec(fig, ax, R, Z, R_uni, Z_uni, BR, Bz, **vec_args)

    ax.set_xlim([0, 0.5 * read_args_r['r_max']])
    ax.set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    ax.set_xlabel(r'$R \: (R_g)$', fontsize=FS)
    ax.set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    ax.set_aspect('equal')

    opt['ha'] = 'left'
    ax.text(0.05, 0.90, r'$\rho / \rho_0$', fontsize=FS+12, transform=ax.transAxes, **opt)
    opt['ha'] = 'right'
    ax.text(0.95, 0.90, fr'$\varphi = {(180./np.pi) * PS.phi[0, 0, 0, idx]:.1f}$' +r'$^{\circ}$', 
            fontsize=FS+12, transform=ax.transAxes, **opt)

fig.tight_layout()
plt.show()

Thus, the ``Phi_Slice`` class allows us to rotate through the data. To conclude this section, let's talk about mirroring. Often, we want to explore how two quantities correlate with one another. For instance, we might want to see a side-by-side of the fluid-frame rest-mass density $\rho$ and the $\sigma$ parameter. Let's use the ``Phi_Slice`` class to make such a mirror plot for the three different simulations we've been studying.

In [ ]:
sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]

# Return to original k_min = 0 and k_max = 1
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

# Plotting information for a mirror plot
plt_args = {
    'cmaps'    : [cmr.prinsenvlag, cmr.torch],
    'FS'       : FS,
    'lim_left' : [1.e-3, 1.e3], 
    'lim_right': [1.e-6, 1.e3],
    'log_left' : True, 
    'log_right': True,
    'fix_pole' : 'constant' 
}

read_args_l = read_args_r.copy()

axes = dict.fromkeys(sims)

figsize = [26.0, 8.0]
fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Only one side is needed for a mirror plot.
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    R, Z = PS_r.calc_RZ(0)
    rho = {
        'r' : PS_r.calc_rho(0)
    }
    sigma = {
        'l' : PS_r.calc_sigma(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    # Changes to the "extend" calls.
    plt_args['extend_left']  = cv.check_extend(sigma['l'], plt_args['lim_left']) 
    plt_args['extend_right'] = cv.check_extend(rho['r'],   plt_args['lim_right'])
    
    Cart.plot_xz_mirror(fig, axes[sim], R, Z, sigma, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$R \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    opt['color'] = 'black'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    opt['color'] = 'white'
    axes[sim].text(0.95, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

    axes[sim].text(0.95, 0.10, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

The real advantage to mirror plots is that vector streamlines can be mirrored as well. Here, we'll draw magnetic-field lines on the left ($\sigma$) panels and flow streamlines on the right (density) panels. Note that the ``plot_xz_mirror_vec`` function is designed to mirror the vector streamlines. To show two different sets of streamlines, use ``plot_xz_vec``.

In [ ]:
# Include vector arguments
vec_args = {
   'dens'               : 2.0,
   'linewidth'          : 0.6,
   'color'              : 'white',
   'broken_streamlines' : True,
   'arrowsize'          : 1.0,
    'cross_pole'        : False # Do not allow streamlines to cross the pole
}

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Only one side is needed for a mirror plot.
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    R, Z = PS_r.calc_RZ(0)
    rho = {
        'r' : PS_r.calc_rho(0)
    }
    sigma = {
        'l' : PS_r.calc_sigma(0)
    }
 
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    # Changes to the "extend" calls.
    plt_args['extend_left']  = cv.check_extend(sigma['l'], plt_args['lim_left']) 
    plt_args['extend_right'] = cv.check_extend(rho['r'],   plt_args['lim_right'])
    
    Cart.plot_xz_mirror(fig, axes[sim], R, Z, sigma, rho, **plt_args)

    # 0 to +40 100 points in each direction
    X_uni, Z_uni = PS_r.calc_XZ_uni(40., 100)
    
    # Return both a left and right side.
    BR,  Bz   = PS_r.calc_B_cyl( 0, lr=True)
    uuR, uuz  = PS_r.calc_uu_cyl(0, lr=True)

    VR = {
        'r' : uuR['r'],
        'l' : BR['l']
    }
    Vz = {
        'r' : uuz['r'],
        'l' : Bz['l']
    }

    # Note that the color argument is now a dictionary. The same is true for the mask keyword.
    vec_args['color'] = {'l' : 'black', 'r' : 'white'}
    Cart.plot_xz_vec(fig, axes[sim], R, Z, X_uni, Z_uni, VR, Vz, **vec_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$R \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    opt['color'] = 'black'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    opt['color'] = 'white'
    axes[sim].text(0.95, 0.90, r'$\rho / \rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

    axes[sim].text(0.95, 0.10, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Wonderful! Clear these figures and we'll move on to the next section. 

### Contours

There are at least 4 common contours drawn on H-AMR plots: 
1. $\sigma$ contours,
2. photosphere contours,
3. zero-Bernoulli surface contours,
4. vector potential contours.

Here, we'll calculate each one and show how we might plot it. First, we'll make a plot of $\sigma$ for each simulation and draw the $\sigma = 1$ contour using the ``plt_args`` dictionary.

In [ ]:
# This allows contour plots to be a standalone section
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# For figure labels (again, remnant of names during my Ph.D.).
opt = dict(ha='left', va='center', color='black',
           bbox=dict(facecolor='none', edgecolor='none'))

sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]

# Load in a single slice of the domain.
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.prinsenvlag,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    sigma = {
        'r' : PS_r.calc_sigma(0),
        'l' : PS_l.calc_sigma(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    # Apply contours
    plt_args['contours_right'] = {
        'quantities' : [sigma['r']], 
        'levels'     : [1.], 
        'colors'     : ['black'], 
        'linewidths' : [2.],
        'linestyles' : ['-']
    }
    plt_args['contours_left'] = plt_args['contours_right'].copy()
    plt_args['contours_left']['quantities'] = [sigma['l']]
    
    Cart.plot_xz(fig, axes[sim], X, Z, sigma, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Note that the inputs to ``plt_args['contours_right']`` and ``plt_args['contours_left']`` are dictionaries of __lists__. The most common mistake using this function is forgetting to pass the arguments as lists. From here, we could add additional contour quantities and levels straightforwardly. Let's make contours of constant $\sigma$ in black and contours of constant $\rho/\rho_0$ in purple. 

In [ ]:
# Plotting information
plt_args = {
    'cmap'     : cmr.prinsenvlag,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    sigma = {
        'r' : PS_r.calc_sigma(0),
        'l' : PS_l.calc_sigma(0)
    }
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    # Apply contours
    plt_args['contours_right'] = {
        'quantities' : [sigma['r'], rho['r']], 
        'levels'     : [[0.1, 1.], [1.e-6, 1.e-3, 1.]], 
        'colors'     : ['black', 'purple'], 
        'linewidths' : [2., 2.],
        'linestyles' : ['-', '-']
    }
    plt_args['contours_left'] = plt_args['contours_right'].copy()
    plt_args['contours_left']['quantities'] = [sigma['l'], rho['l']]
    
    Cart.plot_xz(fig, axes[sim], X, Z, sigma, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Again, this is pretty chaotic and thus has questionable scientific value. One area of scientific value may come from drawing contours to break up the colormap. We'll do that here.

In [ ]:
# Plotting information
plt_args = {
    'cmap'     : cmr.prinsenvlag,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    sigma = {
        'r' : PS_r.calc_sigma(0),
        'l' : PS_l.calc_sigma(0)
    }
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    # Apply contours
    plt_args['contours_right'] = {
        'quantities' : [sigma['r']], 
        'levels'     : [[1.e-3, 0.01, 0.1, 1., 10.]], 
        'colors'     : ['black'], 
        'linewidths' : [2.],
        'linestyles' : ['-']
    }
    plt_args['contours_left'] = plt_args['contours_right'].copy()
    plt_args['contours_left']['quantities'] = [sigma['l'], rho['l']]
    
    Cart.plot_xz(fig, axes[sim], X, Z, sigma, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Because contours of $\sigma$ are so commonly used in H-AMR, we already have built-in functions to handle that. Let's apply it here.

In [ ]:
# Plotting information
plt_args = {
    'cmap'     : cmr.prinsenvlag,
    'FS'       : FS,
    'lim'      : [1.e-3, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    sigma = {
        'r' : PS_r.calc_sigma(0),
        'l' : PS_l.calc_sigma(0)
    }
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    # Apply contours
    # No need for photospheres yet but this argument must be passed.
    photospheres = {
        'scattering' : False,
        'effective'  : False
    }
    colors = ['black']
    sigma_contour = [1.e-3, 1.e-2, 0.1, 1, 10.]
    plt_args = cv.get_contours(plt_args, PS_r, PS_l, colors, sigma_contour, photospheres)
        
    Cart.plot_xz(fig, axes[sim], X, Z, sigma, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\sigma$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Let's move on to photospheres. There is no meaningful density value in simulations without radiation and/or two-temperature effects since physical cross-sections only enter into the problem via scatter/absorption/emission opacities and/or Coulomb-collisional cross-sections. Thus, we're only going to plot the SANE and MAD with cgs units applied. We'll plot the density with the scattering photosphere shown with a white solid line and the effective absorption photosphere shown with a white dashed line. As far as definitions for these quantities, we'll leave those for Part II of this tutorial. 

In [ ]:
# Load in a single slice of the domain.
read_args_r = {
    'cgs_grid' : True,
    'cgs_data' : True,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)

Ah my penchant for pedagogy got the best of me. Remember that whenever loading in the grid with CGS units, you must define a black hole mass ``MBH``. 

In [ ]:
# Load in a single slice of the domain.
read_args_r = {
    'cgs_grid' : True,
    'cgs_data' : True,
    'MBH'      : 10.,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-9, 1.e-1],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 2, figure = fig)
for i, sim in enumerate(sims[1:]):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims[1:], file_numbers[1:]):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    sigma = {
        'r' : PS_r.calc_sigma(0),
        'l' : PS_l.calc_sigma(0)
    }
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    # Apply contours
    # Calculate the theta-integrated photosphere.
    photospheres = {
        'scattering' : True,
        'effective'  : True,
        'integral'   : 'theta'
    }
    colors = ['black', 'white', 'cyan'] # The ordering of the colors is sigma contours, scattering photosphere, then effective absorption photosphere
    sigma_contour = False
    plt_args = cv.get_contours(plt_args, PS_r, PS_l, colors, sigma_contour, photospheres)
        
    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    opt['color'] = 'white'
    axes[sim].text(0.05, 0.90, r'$\rho \: \left(\rm{g/cm}^3 \right)$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Nice! We have photospheres! Let's now calculate the zero-Bernoulli surface. 

In [ ]:
# Load in a single slice of the domain.
read_args_r = {
    'cgs_grid' : True,
    'cgs_data' : True,
    'MBH'      : 10.,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-9, 1.e-1],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 2, figure = fig)
for i, sim in enumerate(sims[1:]):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims[1:], file_numbers[1:]):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    Be = {
        'r' : PS_r.calc_Bernoulli(0),
        'l' : PS_l.calc_Bernoulli(0)
    }
    
    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(rho['r'], plt_args['lim']) 

    # Apply contours
    # Calculate the theta-integrated photosphere.
    photospheres = {
        'scattering' : False,
        'effective'  : False,
        'integral'   : 'theta'
    }
    colors = ['black', 'white', 'cyan'] # The ordering of the colors is sigma contours, scattering photosphere, then effective absorption photosphere
    sigma_contour = False
    plt_args = cv.get_contours(plt_args, PS_r, PS_l, colors, sigma_contour, photospheres)

    plt_args['contours_right'] = {
        'quantities' : [Be['r']],
        'levels'     : [0.],
        'colors'     : ['white'],
        'linewidths' : [2.]
    }
    
    plt_args['contours_left'] = {
        'quantities' : [Be['l']],
        'levels'     : [0.],
        'colors'     : ['white'],
        'linewidths' : [2.]
    }    

    Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    opt['color'] = 'white'
    axes[sim].text(0.05, 0.90, r'$\rho \: \left(\rm{g/cm}^3 \right)$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Excellent! Last but certainly not least, we want to calculate vector potential ($A_{\phi}$) contours. We will use the function ``compute_variables.get_Aphi_contours``.

In [ ]:
# Load in a single slice of the domain.
read_args_r = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'MBH'      : 10.,
    'r_max'    : 80.,
    'k_min'    : 0,
    'k_max'    : 1
}

read_args_l = read_args_r.copy()

# Plotting information
plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True,
    'fix_pole' : 'constant' 
}

axes = dict.fromkeys(sims)

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
for i, sim in enumerate(sims):
    axes[sim] = fig.add_subplot(gs[0, i])

Cart = figs.Cartesian([]) # Cartesian class takes params as an argument but currently this argument is not used

for sim, file_number in zip(sims, file_numbers):
    # Right side
    PS_r = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_r)
    X, Z = PS_r.calc_RZ(0)
    ipi  = int(PS_r.nphi / 2)
    # Left side
    read_args_l['k_min'], read_args_l['k_max'] = ipi, ipi + 1
    PS_l = cv.Phi_Slice(f'{cv.shared_path}/{sim}', file_number, **read_args_l)
    # Both sides
    rho = {
        'r' : PS_r.calc_rho(0),
        'l' : PS_l.calc_rho(0)
    }

    plt_args['r']     = PS_r.r
    plt_args['theta'] = PS_r.th
    plt_args['phi']   = PS_r.phi

    plt_args['extend'] = cv.check_extend(sigma['r'], plt_args['lim']) 

    im, _ = Cart.plot_xz(fig, axes[sim], X, Z, rho, **plt_args)

    # Apply contours
    plt_args_Aphi = plt_args.copy()
    contour_colors = ['white', 'white', 'cyan']
    plt_args_Aphi = cv.get_Aphi_contours(plt_args_Aphi, PS_r, PS_l, contour_colors)

    Xb, Zb = PS_r.calc_XZ_base(0)
    Cart.plot_contours(axes[sim], im['l'], Xb['l'], Zb['l'], plt_args_Aphi['contours_left'])
    Cart.plot_contours(axes[sim], im['r'], Xb['r'], Zb['r'], plt_args_Aphi['contours_right'])

    axes[sim].set_xlim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_ylim([-0.5 * read_args_r['r_max'], 0.5 * read_args_r['r_max']])
    axes[sim].set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    axes[sim].set_ylabel(r'$z \: (R_g)$', fontsize=FS)
    axes[sim].set_aspect('equal')

    opt['ha'] = 'left'
    axes[sim].text(0.05, 0.90, r'$\rho/\rho_0$', 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)
    opt['ha'] = 'right'
    axes[sim].text(0.95, 0.90, sim_names[sim], 
                   fontsize=FS+12, transform=axes[sim].transAxes, **opt)

fig.tight_layout()
plt.show()

Note that we needed to calculate the base grid coordinates and then plot contours. In order to plot on the same image, we used the fact that ``Cart.plot_xz`` returns an image dictionary. For future reference, you should be able to make modifications to figures using the objects returned; however, in general this will not be necessary. Note too that the color used to highlight the $\Phi = 50$ contour is the third in the list. You could just as easily have made this color "white" to make all of the contours look similar. 

### By Block Plots

While ``Midplane_Slice`` and ``Phi_Slice`` allow you to handle very small slices into high-resolution data, one can load in the global domain and easily create figures using the same call structure as the original toolkit. Here, we'll show how to work with the ``HAMR_Cartesian`` and ``HAMR_Cartesian_Block`` classes. We produce the same midplane and poloidal plots for the ``T65MAD`` simulation. Note, however, that while vector plots are available to ``HAMR_Cartesian``, they must be called in the same way as was done for the ``Cartesian`` class, since ``HAMR_Cartesian`` inherits, but does not override, the vector plotting routines from ``Cartesian``.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# Load in full domain.
read_args = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'by_block' : False,
    'r_max'    : 80.
}

H = hr.Read(f'{cv.shared_path}/T65MAD', **read_args)
H.read_grid(1500, direct=True)
H.read_prims(1500, skip_read_params=True)

plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True, 
}

Cart = figs.HAMR_Cartesian() # No argument needs to be passed here

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
ax1  = fig.add_subplot(gs[0, 0])
ax2  = fig.add_subplot(gs[0, 1])
ax3  = fig.add_subplot(gs[0, 2])

plt_args['fix_pole'] = 'average'
Cart.plot_xy(fig, ax1, H.data['rho'], **H.grid, **H.data, **plt_args)
plt_args['fix_pole'] = 'constant'
Cart.plot_xz(fig, ax2, H.data['rho'], **H.grid, **H.data, **plt_args)
Cart.plot_Rz(fig, ax3, H.data['rho'], **H.grid, **H.data, **plt_args)

for ax in [ax1, ax2, ax3]:
    ax.set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    ax.set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    ax.set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    ax.set_aspect('equal')

ax3.set_ylabel(r'$R \: (R_g)$', fontsize=FS)

ax1.set_ylabel(r'$y \: (R_g)$', fontsize=FS)
ax2.set_ylabel(r'$z \: (R_g)$', fontsize=FS)
ax3.set_ylabel(r'$z \: (R_g)$', fontsize=FS)

ax3.set_xlim([0., read_args['r_max']])

fig.tight_layout()
plt.show()

Plotting the block structure is similarly quite straightforward using the same call structure. At the time of writing, vector plots "by block" are not included. This is on the to-do list. The next section may be memory intensive so if the kernel crashes, try clearing the outputs of all cells first.

Note that I use the ``show_grid`` parameter in ``plt_args``. This is a downsampling parameter for the true grid lines. If ``show_grid`` is used and set to 1, the true grid will be shown. If ``show_grid=2``, every other grid line will be drawn.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import figs # Figure-making functions

figsize = [32.0, 18.0]
FS      = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS

# Load in full domain.
read_args = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'by_block' : True,
    'r_max'    : 80., # this is meaningless with by_block=True
}

H = hr.Read(f'{cv.shared_path}/T65MAD', **read_args)
H.read_grid(1500, direct=True)
H.read_prims(1500, skip_read_params=True)

plt_args = {
    'cmap'     : cmr.torch,
    'FS'       : FS,
    'lim'      : [1.e-6, 1.e3],
    'log'      : True, 
    'show_grid': 4
}

Cart = figs.HAMR_Cartesian_Block(H.params) # Note that for HAMR_Cartesian_Block, there needs to be a valid parameters dictionary inserted here.

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 3, figure = fig)
ax1  = fig.add_subplot(gs[0, 0])
ax2  = fig.add_subplot(gs[0, 1])
ax3  = fig.add_subplot(gs[0, 2])

#plt_args['fix_pole'] = 'average'
Cart.plot_xy(fig, ax1, H.data['rho'], **H.grid, **H.data, **plt_args)
#plt_args['fix_pole'] = 'constant'
Cart.plot_xz(fig, ax2, H.data['rho'], **H.grid, **H.data, **plt_args)
Cart.plot_Rz(fig, ax3, H.data['rho'], **H.grid, **H.data, **plt_args)

for ax in [ax1, ax2, ax3]:
    ax.set_xlim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    ax.set_ylim([-0.5 * read_args['r_max'], 0.5 * read_args['r_max']])
    ax.set_xlabel(r'$x \: (R_g)$', fontsize=FS)
    ax.set_aspect('equal')

ax3.set_ylabel(r'$R \: (R_g)$', fontsize=FS)

ax1.set_ylabel(r'$y \: (R_g)$', fontsize=FS)
ax2.set_ylabel(r'$z \: (R_g)$', fontsize=FS)
ax3.set_ylabel(r'$z \: (R_g)$', fontsize=FS)

ax3.set_xlim([0., read_args['r_max']])

fig.tight_layout()
plt.show()

Wonderful! The thicker lines will show you block boundaries while the thinner lines show the grid structure. You can change the value of the phi slice that you're plotting using the ``phi0`` paramter, and you can set the ``grid_color`` and the linewidth of the grids (``grid_width``) using the appropriate parameters passed as ``plt_args``. You now have the tools you need to start exploring the H-AMR data. Before moving on, you should clear the outputs to keep the kernel in tact.

### Creating Custom Images

While many of the quantities that would be of interest are calculated in the ``Phi_Slice``class, you may find a quantity that you'd like to plot but which is not currently implemented. Let's work through an example. Suppose that we wanted the visualize $\rho v^r$, where $v^r = u^r / u^t$ is the 3-velocity. I don't think that quantity is in ``Phi_Slice``, so as a user, the best way to add this in would be to create a ``My_Phi_Slice`` class that inherits ``Phi_Slice``. Let's see that in action. 

In [ ]:
class My_Phi_Slice(cv.Phi_Slice):
    def __init__(self, directory, fnum, **kwargs):
        super().__init__(sirectory, fnum, **kwargs)

    def calc_rhovr(self, iphi):
        # Fetch variables in (x1, x2) coordinates.
        rho, uu = self.get_vars2D(['rho', 'uu'], iphi)
        
        # Initialize coordinate transformations.
        self.set_Coords2D(iphi)

        # Calculate uu in spherical coordinates
        uu_sph = self.Coords2D.int_to_sph_vec(uu)
        vr     = self.c * (uu_sph[1] / uu_sph[0])
        
        return rho * vr



This completes Part I of the tutorial. Please clear the outputs of all cells to keep the kernel from crashing. We'll move on now to the analysis.

## II) Analysis

The analysis routines for the H-AMR Visualization toolkit are distributed across 5 different files, all contained within the ``HAMR_Visualization/Analyze_HAMR_Data`` directory:

1. ``analysis.py``: contains ``Accretion``, ``Field``, ``Fluid``, ``MRI``, and ``Tools`` classes related to calculating mass/energy/angular momentum fluxes, magnetic fluxes, stress-energy and electromagnetic field-strength tensors, and tools related to tensor constructions and manipulations.
2. ``coordinates.py``: contains ``Coordinates``, ``Integrals``, ``Kerr_Schild``, ``Tetrads``, ``Tilt``, and ``Christoffel_KS`` classes related to coordinate transformations, integrals, and calculations of the angular-momentum tensor for calculating tilt and precession angles.
3. ``radiation.py``: contains ``Radiation`` and ``Opacity`` classes for calculating the radiation stress-energy tensor and scatter/emission/absorption opacities.
4. ``theory.py``: contains the ``Analytic``, ``Novikov_Thorne``, ``Shakura_Sunyaev``, and ``ADAF`` classes, which include various analytic results such as the Eddington luminosity, various radii in the Kerr metric, the Novikov & Thorne (1973) dissipation rate, and analytic results related to thin disks and ADAFs.

### theory.py

The Eddington luminosity is defined 

$$ L_{\rm Edd} = 1.26 \times 10^{38} \left( \frac{M_{\rm BH}}{M_{\odot}} \right) \: \rm{erg/s}, $$

and the Eddington mass-accretion rate is defined

$$ \dot{M}_{\rm Edd} = L_{\rm Edd} / \eta c^2 \: \rm{g/s}, $$

where $\eta$ is the efficiency of energy release for the accretion flow. We can calculate $\eta$ based on the Novikov & Thorne (1973) dissipation rate using the ``Analytic`` class.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import theory

MBH = 10. # 10 Msun black hole
a   = 0.94 # typical black hole spin used in H-AMR
Analytic = theory.Analytic(hr.CONSTANTS, cgs=True)

r = np.logspace(0, 3, 1000)

L_Edd = Analytic.calc_LEdd(MBH)
eta = Analytic.calc_efficiency_NT73(r, a)
deta_dr = Analytic.calc_dissipation_NT73(r, a)

eta0 = eta[r >= 200][0]

Mdot_Edd = Analytic.calc_Mdot_Edd(MBH, eta=eta0)

print(f"Eddington luminosity for {MBH} solar mass black hole: {L_Edd:.2e} erg/s")
print(f"Eddington mass-accretion rate for {MBH} solar mass black hole assuming Novikov & Thorne (1973) efficiency: {Mdot_Edd:.2e} g/s")

opt = dict(ha='left', va='center', color='black',
           bbox=dict(facecolor='None', edgecolor='none'))

FS = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS
figsize = [8.0, 10.0]

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(2, 1, figure = fig)
ax1  = fig.add_subplot(gs[0, 0])
ax2  = fig.add_subplot(gs[1, 0])

ax1.plot(r, deta_dr, lw=2, color='black')
ax2.plot(r, eta, lw=2, color='black')

ax1.set_ylabel(r'$d \eta / d r$', fontsize=FS)
ax2.set_ylabel(r'$\eta$',         fontsize=FS)

for ax in [ax1, ax2]:
    ax.set_xlabel(r'$r / R_g$', fontsize=FS)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim([1, 100])
    ax.set_xticks([1, 2, 10, 50, 100])
    ax.set_ylim([1.e-3, 1.])
    ax.set_xticklabels(['1', '2', '10', '50', '100'])
    cv.grid(ax, alpha=0.2, color='black')
    
ax2.text(0.05, 0.90, r'$a = %.4f$' %a, fontsize=FS, transform=ax2.transAxes, **opt)
ax2.text(0.05, 0.80, r'$\eta (r/R_g = 200) = %.3f$' %eta[r >= 200.][0], fontsize=FS, transform=ax2.transAxes, **opt)
ax1.text(0.05, 0.90, r'$\text{Novikov & Thorne (1973)}$', fontsize=FS, transform=ax1.transAxes, **opt)
fig.tight_layout()
plt.show()

The angular velocity of a particle in orbit around a 10 solar mass black hole with spin parameter $a$ is

$$ \Omega = \left( \frac{G M_{\rm BH}}{R_g^3} \right)^{1/2} \left( (r/R_g)^{3/2} + a  \right)^{-1} \: \rm{s}^{-1}. $$

The outer horizon of a Kerr black hole is

$$ r_H = R_g \left( 1 + \sqrt{1 - a^2} \right). $$

We show a comparison of the non-relativistic Keplerian and Kerr-metric angular velocity in the next figure. The solid vertical line shows the location of the outer event horizon while the dashed line provides the radius of the innermost stable circular orbit.

In [ ]:
G    = hr.CONSTANTS['G']
c    = hr.CONSTANTS['c']
Msun = hr.CONSTANTS['Msun']
Rg   = G * MBH * Msun / (c * c)

# Orbital frequencies
OmegaK_relativistic = Analytic.calc_OmegaK(MBH, a, r)
OmegaK              = np.sqrt(G * MBH * Msun / (r * Rg) ** 3)

# Horizon radius
rH    = Analytic.calc_R_Horizon(a)
rISCO = Analytic.calc_R_ISCO(a) 

FS = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS
figsize = [6.0, 6.0]

fig = plt.figure(figsize = figsize)
gs  = gridspec.GridSpec(1, 1, figure = fig)
ax  = fig.add_subplot(gs[0, 0])

ax.plot(r, OmegaK,              lw=2, color='black',      label=r'$\text{non-relativistic}$')
ax.plot(r, OmegaK_relativistic, lw=2, color='dodgerblue', label=r'$\text{Kerr black hole}$')
ax.axvline(x=rH,    color='black', lw=2)
ax.axvline(x=rISCO, color='black', lw=2, linestyle='--')

ax.set_ylabel(r'$\Omega_{\rm K} \: (\rm{s}^{-1})$', fontsize=FS)

ax.set_xlabel(r'$r / R_g$', fontsize=FS)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim([1, 100])
cv.grid(ax, alpha=0.2, color='black')

fig.tight_layout()
plt.show()

### C++ Acceleration

Most performance-critical functions in the toolkit have a C++-accelerated version that is called under the hood if you pass ``cpp=True``. In general, if working with full 3D data objects, you should activate the C++ acceleration, which will take advantage of multithreading and usually has a lower memory footprint. However, for smaller 1D or 2D objects, often the overhead of calling to the C++ functions is unnecessary, and the Python functions are completely sufficient. Rarely will you have to call C++ functions directly. Rather, you will call a function like ``Field.calc_TEM(...necessary arguments..., cpp=True)``.

### analysis.py

#### Accretion

The main analysis can be found in the ``Accretion`` class. We'll demonstrate each of the options used in this class for calculating the mass-accretion rate; however, the same structure applies to the energy-accretion rate and the angular-momentum-accretion rate.

We define the mass-accretion rate $\dot{M}$ such that a positive $\dot{M}$ implies that material is moving toward the black hole,

$$ \dot{M} = -R_g^2 c \int \int \rho u^r \sqrt{-g} d \theta d \varphi, $$

and the energy-accretion rate $\dot{E}$ is defined such that a positive $\dot{E}$ implies that energy is flowing radially inward toward the black hole,

$$ \dot{E} = R_g^2 c \int \int T^{r}_{t, \rm{MHD}} \sqrt{-g} d \theta d \varphi, $$

where the full MHD stress-energy tensor (including radiation when radiation transport is enabled in the code) is given by

$$ T^{\mu}_{\nu, \rm{MHD}} = T^{\mu}_{\nu, \rm{HD}} + T^{\mu}_{\nu, \rm{EM}} + R^{\mu}_{\nu} = 
\left( \rho c^2 + P_{\rm gas} + \mathcal{E}_{\rm gas} \right) \frac{u^{\mu}}{c} \frac{u_{\nu}}{c} + P_{\rm gas} \delta^{\mu}_{\nu} + 
\frac{b^2}{4 \pi} \frac{u^{\mu}}{c} \frac{u_{\nu}}{c} + \frac{b^2}{8 \pi} \delta^{\mu}_{\nu} - \frac{b^{\mu} b_{\nu}}{4 \pi} + 
\frac{4}{3} \mathcal{E}_{\rm rad} \frac{u^{\mu}_{\rm rad}}{c} \frac{u_{\nu, \rm{rad}}}{c} + \frac{1}{3} \mathcal{E}_{\rm rad} \delta^{\mu}_{\nu} = 
\left( \rho c^2 + P_{\rm gas} + \mathcal{E}_{\rm gas} + \frac{b^2}{4 \pi} \right) \frac{u^{\mu}}{c} \frac{u_{\nu}}{c} + \left(P_{\rm gas} + \frac{b^2}{8 \pi} \right) \delta^{\mu}_{\nu} - \frac{b^{\mu} b_{\nu}}{4 \pi} + \frac{4}{3} \mathcal{E}_{\rm rad} \frac{u^{\mu}_{\rm rad}}{c} \frac{u_{\nu, \rm{rad}}}{c} + \frac{1}{3} \mathcal{E}_{\rm rad} \delta^{\mu}_{\nu}. $$

Here, "HD" refers to hydrodynamic or fluid, "EM" refers to electromagnetic, and $\mathcal{R}^{\mu}_{\nu}$ is the radiation stress-energy tensor. Finally, the angular-momentum-accretion rate is defined

$$ \dot{L} = R_g^2 c \int \int T^{r}_{\varphi, \rm{MHD}} \sqrt{-g} d \theta d \varphi. $$

The following "cuts" are available on these mass, energy, and angular-momentum inflow/outflow rates:
1. ``sigma_cut``: removes all data with $\sigma$ > ``sigma_cut``.
2. ``Temp_cut`` : if ``cold=True``, removes all data with temperature (the ion temperature, $T_i$, if a temperature ``Temp`` is not provided) > ``Temp_cut``. If ``hot=True`` then hot material is maintained.
3. ``HoR_cut``: removes material based on the local disk thickness $H_{\rho}/R$, keeping material within the disk unless ``above=True``.
4. ``in`` and ``out``: keeps material that is either flowing "in" or "out" based on the value of ``uu[1]``.
5. ``bound`` and ``unbound``: keeps bound or unbound material based on the value of the Bernoulli parameter (not including radiation, to include radiation you must pass in a custom value for ``Be``.
6. ``tau_cut``: removes material based on optical depth $\tau$. If ``optically_thin=True`` (default), then optically thin ($\tau < 1$) material will be maintained. Otherwise, optically thick ($\tau > 1$) material will be selected. 

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import analysis as ana

# Calculate and plot mass-accretion rate for all simulations.

sims      = ['T65MAD', 'TRANS_XRB_HIGH_SANE', 'TRANS_XRB_HIGH_MAD']
sim_names = {
    'T65MAD'              : r'$\text{Tilted MAD}$', 
    'TRANS_XRB_HIGH_SANE' : r'$\text{SANE}$', 
    'TRANS_XRB_HIGH_MAD'  : r'$\text{MAD}$'
}
file_numbers = [1500, 9000, 9000]
H = dict.fromkeys(sims)
# Downsample for speed.
read_args = {
    'cgs_grid' : False,
    'cgs_data' : False,
    'lowres1'  : 4,
    'lowres2'  : 4,
    'lowres3'  : 4
}

Mdot = dict.fromkeys(sims)
Acc  = dict.fromkeys(sims)

for sim, file_number in zip(sims, file_numbers):
    H[sim] = hr.Read(f'{cv.shared_path}/{sim}', **read_args)
    H[sim].read_grid(file_number, direct=True)
    H[sim].read_prims(file_number, skip_read_params=True)

    acc_class_args = {
        'grid'      : H[sim].grid,
        'data'      : H[sim].data,
        'pars'      : H[sim].params,
        'CONSTANTS' : hr.CONSTANTS,
        'cgs'       : False
    }

    # Initialize Accretion class for each simulation.
    Acc[sim] = ana.Accretion(**acc_class_args)

    # Calculate the mass-accretion rate.
    Mdot[sim] = Acc[sim].calc_Mdot()[0] # note the block index

In [ ]:
import cmasher as cmr # colormaps I like
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Required to plot horizon and ISCO radius.
import theory
MBH = 10. # 10 Msun black hole
a   = 0.94 # typical black hole spin used in H-AMR
Analytic = theory.Analytic(hr.CONSTANTS, cgs=True)

colors = {
    'T65MAD'              : 'black',
    'TRANS_XRB_HIGH_SANE' : 'dodgerblue',
    'TRANS_XRB_HIGH_MAD'  : 'darkorange'
}

FS = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS
figsize = [8.0, 6.0]

fig = plt.figure(figsize = figsize)
gs  = gridspec.GridSpec(1, 1, figure = fig)
ax  = fig.add_subplot(gs[0, 0])

for sim in sims:
    r = H[sim].grid['r'][0, :, 0, 0]
    ax.plot(r, Mdot[sim], lw=2, color=colors[sim], label=sim_names[sim])

    # Horizon radius
    rH    = Analytic.calc_R_Horizon(H[sim].params['a'])
    rISCO = Analytic.calc_R_ISCO(H[sim].params['a']) 

ax.axvline(x=rH,    color='black', lw=2)
ax.axvline(x=rISCO, color='black', lw=2, linestyle='--')

ax.set_ylabel(r'$\dot{M} \: (R_g^2 \rho_0 c)$', fontsize=FS)

ax.set_xlabel(r'$r / R_g$', fontsize=FS)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim([1, 1000])
cv.grid(ax, alpha=0.2, color='black')
ax.set_ylim([1.e-4, 1.e4])

fig.tight_layout()
plt.show()

We now demonstrate various cuts on the mass-accretion rate.

In [ ]:
Mdot_sigma = dict.fromkeys(sims)
Mdot_disk  = dict.fromkeys(sims)

for sim, file_number in zip(sims, file_numbers):
    # Calculate the mass-accretion rate by applying a sigma cut.
    Mdot_sigma[sim] = Acc[sim].calc_Mdot(sigma_cut=1.)[0] # note the block index
    Mdot_disk[sim]  = Acc[sim].calc_Mdot(HoR_cut=1.)[0]

In [ ]:
fig = plt.figure(figsize = figsize)
gs  = gridspec.GridSpec(1, 1, figure = fig)
ax  = fig.add_subplot(gs[0, 0])

for sim in sims:
    r = H[sim].grid['r'][0, :, 0, 0]
    #ax.plot(r, Mdot[sim], lw=2, color=colors[sim], label=sim_names[sim], linestyle='-')
    #ax.plot(r, Mdot_sigma[sim], lw=2, color=colors[sim], label=sim_names[sim], linestyle='--')
    ax.plot(r, Mdot_disk[sim], lw=2, color=colors[sim], label=sim_names[sim], linestyle='-.')
    

    # Horizon radius
    rH    = Analytic.calc_R_Horizon(H[sim].params['a'])
    rISCO = Analytic.calc_R_ISCO(H[sim].params['a']) 

ax.axvline(x=rH,    color='black', lw=2)
ax.axvline(x=rISCO, color='black', lw=2, linestyle='--')

ax.set_ylabel(r'$\dot{M} \: (R_g^2 \rho_0 c)$', fontsize=FS)

ax.set_xlabel(r'$r / R_g$', fontsize=FS)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim([1, 1000])
cv.grid(ax, alpha=0.2, color='black')
ax.set_ylim([1.e-4, 1.e4])

fig.tight_layout()
plt.show()

#### Field

Quantities related to the magnetic field are handled by the ``Field`` class. If ``cgs=True`` is passed, then all functions in the ``Field`` class assume that the arguments have units. The one exception is when passing ``uu_tnsr`` in the ``Field.calc_TEM`` function since $u^{\mu} u^{\nu}$ (if the default ``upper_upper=True``) is assumed to be a dimensionless dyadic tensor. See GitHub documentation for more details on the ``Field`` class. Here, we test the individual functions to demonstrate their call structure.

In [29]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

import analysis as ana

read_args = {
    'cgs_data' : True, 
    'cgs_grid' : True,
    'lowres1'  : 2,
    'lowres2'  : 2,
    'lowres3'  : 2,
    'MBH'      : 10.,
    'coords1D' : True
}

H = hr.Read(f'{cv.shared_path}/TRANS_XRB_HIGH_SANE', **read_args)
H.read_grid(9000, direct=True)
H.read_prims(9000, skip_read_params=True)

class_args = {
    'pars'      : H.params,
    'CONSTANTS' : hr.CONSTANTS,
    'cgs'       : True
}

Field = ana.Field(**class_args)

# 4-vector magnetic field
b        = Field.calc_b(**H.data, **H.grid)
# b^2
b2_fromB = Field.calc_b2(b=H.data['B'], **H.data, **H.grid, from_B=True, cpp=True)
b2       = Field.calc_b2(b=b, **H.data, **H.grid, cpp=True)
# Electromagnetic stress-energy tensor
TEM_t1   = Field.calc_TEM(**H.data, **H.grid, upper_lower=True, index=[1, 0])
# Field-strength tensor and its dual
FEM      = Field.calc_FEM(**H.data, **H.grid)
dual_FEM = Field.calc_dual_FEM(**H.data, **H.grid)
# Current density
J        = Field.calc_J(**H.data, **H.grid)
# J^2
J2_fromF = Field.calc_J2(J=H.data['uu'],  **H.data, **H.grid, from_FEM=True)
J2       = Field.calc_J2(J=J, **H.data, **H.grid)
# Alfven velocity
vA       = Field.calc_vA(**H.data, **H.grid)

b2_diff = np.sum(np.abs(b2 - b2_fromB)) / np.sum(b2)
J2_diff = np.sum(np.abs(J2 - J2_fromF)) / np.sum(J2)
print(f"The relative difference between the two ways of calculating b2 is {b2_diff}.")
print(f"The relative difference between the two ways of calculating J2 is {J2_diff}.")

Loading primitive variables for /lustre/orion/ast178/proj-shared/TRANS_XRB_HIGH_SANE/dumps9000
The relative difference between the two ways of calculating b2 is 1.1160322088699104e-07.
The relative difference between the two ways of calculating J2 is 0.0.


#### Fluid

Quantities related to two-temperature effects and the hydrodynamic "fluid" are handled by the ``Fluid`` class. If ``cgs=True`` is passed, then all functions in the ``Fluid`` class assume that the arguments have units. The one exception is when passing ``uu_tnsr`` in the ``Fluid.calc_THD`` function. Here, we test the individual functions to demonstrate their call structure. The Coulomb coupling rate, thermal timescales, and the differential ion-electron heating fraction are contained in the ``Fluid`` class. 

In [35]:
class_args = {
    'pars'      : H.params,
    'CONSTANTS' : hr.CONSTANTS,
    'cgs'       : True
}

Fluid = ana.Fluid(**class_args)

# Gas thermal pressure (variable adiabatic index)
pgas  = Fluid.calc_pgas(**H.data)

# Gas thermal pressure (constant adiabatic index)
gamma_adi = H.params['gam']
pgas      = (gamma_adi - 1.) * H.data['u_gas']

# Stress-energy tensor
THD = Fluid.calc_THD(**H.data, **H.grid, pgas=pgas)

# Timescales
teq   = Fluid.calc_teq(**H.data)
tii = Fluid.calc_tii(**H.data)
tee = Fluid.calc_tee(**H.data)
qie = Fluid.calc_qie(**H.data)

# Differential electron heating fraction
delta_e = Fluid.calc_delta_e(**H.data, **H.grid)

print(delta_e.shape)

TypeError: Fluid.calc_pgas() missing 2 required positional arguments: 'Te' and 'Ti'

#### MRI

Quantities related to properties of the magnetorotational instability (MRI) are contained in the ``MRI`` class. The most important quantities in this class are the MRI quality factors, which are used to assess if resolution is adequate in simulations. Note that these MRI quality factors are based on the __true resolution__ of the simulation. They are not based on the upscaling to a uniform grid. We demonstrate the ``MRI`` class here.

### coordinates.py

#### Tilt

The ``Tilt`` class is used to calculate the angular momentum, and the tilt and precession angles of an accretion flow. It also contains the methods to remap the flow to a tilted coordinate system. We'll explore some examples. 

Following Fragile & Anninos (2005), we define the angular momentum tensor,

$$ L^{\mu \nu} (r) = \int \int \left( x^{\mu} T_{\rm Cart}^{\nu 0} - x^{\nu} T_{\rm Cart}^{\mu 0} \right) \sqrt{-g} d x_2 d x_3, $$

where the hydrodynamic/fluid stress-energy tensor in a Cartesian basis is

$$ T^{\mu \nu}_{\rm Cart} = \left( \rho c^2 + \mathcal{E}_{\rm gas} + P_{\rm gas} \right) \frac{u^{\mu}_{\rm Cart}}{c} \frac{u^{\nu}_{\rm Cart}}{c} + P_{\rm gas} g^{\mu \nu}_{\rm Cart}, $$ 

and $x^{\mu} = (t, x, y, z)$ is the position 4-vector in Cartesian coordinates. We can calculate this tensor using the ``Tilt`` class.

In [33]:
# This is a standalone section (memory intensive)
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

read_args = {
    'cgs_data' : False, 
    'cgs_grid' : False,
    'lowres1'  : 8,
    'lowres2'  : 8,
    'lowres3'  : 8
}

H = hr.Read(f'{cv.shared_path}/T65MAD', **read_args)
H.read_grid(100, direct=True)
H.read_prims(100, skip_read_params=True)

Loading primitive variables for /lustre/orion/ast178/proj-shared/T65MAD/dumps100


In [34]:
import analysis as ana
import coordinates as coords

tilt_class_args = {
    'grid'      : H.grid,
    'data'      : H.data,
    'pars'      : H.params,
    'CONSTANTS' : hr.CONSTANTS
}

class_args = {
    'pars'      : H.params,
    'CONSTANTS' : hr.CONSTANTS
}

# Initialize Accretion class (you can use the tilt class arguments since Tilt inherits Accretion)
Acc = ana.Accretion(**tilt_class_args)
# Initialize Fluid class
Fluid = ana.Fluid(**class_args)

gamma_adi = 5. / 3.
pgas = (gamma_adi - 1.) * H.data['u_gas']

Tilt = coords.Tilt(**tilt_class_args)
Lmn  = Acc.thph_sum(Tilt.calc_angular_momentum_tnsr(), sigma_cut=1.)

THD_vec    = np.zeros_like(H.data['uu'])
THD_vec[0] = Fluid.calc_THD(**H.data, **H.grid, pgas=pgas, index=[0, 0], cpp=True)
THD_vec[1] = Fluid.calc_THD(**H.data, **H.grid, pgas=pgas, index=[1, 0], cpp=True)
THD_vec[2] = Fluid.calc_THD(**H.data, **H.grid, pgas=pgas, index=[2, 0], cpp=True)
THD_vec[3] = Fluid.calc_THD(**H.data, **H.grid, pgas=pgas, index=[3, 0], cpp=True)

THD_cart_vec = Tilt.int_to_cart_vec(THD_vec, cpp=True)

Svec      = Acc.thph_sum(THD_cart_vec, sigma_cut=1.)
gcov_cart = Tilt.int_to_cart_cotnsr(H.grid['gcov'])
S_cov     = Acc.thph_sum(np.einsum('ij...,j...->i...', gcov_cart, THD_cart_vec), sigma_cut=1.)

# Calculate J angular momentum vector
levi_civita = ana.Tools.calc_Levi_Civita(4, np.ones_like(Svec[0]), cov=True)

S2   = np.einsum('i...,i...->...', S_cov, Svec)
Jvec = np.einsum('ijkl...,ij...,k...->l...', levi_civita, Lmn, Svec) / (2. * np.sqrt(-S2))
inclination = np.degrees(np.arccos(Jvec[3] / np.sqrt(Jvec[1] * Jvec[1] + Jvec[2] * Jvec[2] + Jvec[3] * Jvec[3])))

print(inclination)

#print("Raw Jvec components:", Jvec[0], Jvec[1], Jvec[2], Jvec[3])

[[      nan 54.958126 59.3256   65.03657  68.91047  78.828224 84.99787
  92.69327  95.498604 96.45312  95.745056 94.86398  94.291794 91.71533
  91.91597  88.70918  88.43494  88.19988  89.94914  89.72962  89.13134
  88.28629  88.5294   88.49745  88.661095 87.987526 88.10031  88.31587
  88.39753  88.768486 88.399155 88.30556  87.66919  87.86445  87.73247
  88.335335 89.037506 88.60252  87.1967   89.77983  96.68572  93.82194
  96.64508  91.5134   90.73478  91.076225 90.49115  89.44113  89.426254
  89.33589  86.442    86.70228  91.283745 89.16473  87.90486  87.97572
  87.302986 84.591125 83.01149  44.76348  77.92801  74.7191   84.05516
  90.67289        nan 91.53984  89.99999  89.99999  89.99999  89.99999
  90.       90.       89.99999  90.       90.       90.       89.99999
  90.       90.       90.       90.       90.       90.       90.
  90.       90.       90.       90.       89.99999  90.       90.
  90.       90.       90.       90.       90.       90.       90.
  90.       90.     

/tmp/ipykernel_282/1835445813.py:43: RuntimeWarning: invalid value encountered in sqrt
  Jvec = np.einsum('ijkl...,ij...,k...->l...', levi_civita, Lmn, Svec) / (2. * np.sqrt(-S2))


## III) Radial Profiles and Time Series

### Extraction

*"Chris, I don't need to see pretty pictures. Show me lines."
-- Eliot Quataert, about 6 months into my Ph.D.*

Thus far, the tools we have presented require full or partial data sets to be read into memory using a single core. Since a single H-AMR data set can be 10+ petabytes upscaled, it is difficult to do direct comparisons to theory or any global integrals on the full data sets using the tools above. The essence of the toolkit is to do analysis "at scale" using all available resources. Radial profiles and time series data are the first quantiative diagnostics that you will likely use when studying a new simulation. Here, we discuss how to extract, load, average, and plot radial profiles.

Two main scripts are available for profile extraction:
1. ``HAMR_Visualization/Analyze_HAMR_Data/radial_profiles.py``
2. ``HAMR_Visualization/Analyze_HAMR_Data/radial_profiles_time.py``

Without the "_time" convention, the script is designed to operate on a single dump at ultra-high-resolution. With the "_time" convention, the script is designed to operate on many dumps at once. It comes down to the resources you have available and what you're trying to accomplish. If you need to process a few very high-resolution data dumps, then ``radial_profiles.py`` is the proper tool to use. If you have many thousands of data files that are not especially large, then ``radial_profiles_time.py`` is more appropriate. Both scripts produce the same outputs; however, they're parallelized differently.

In ``radial_profiles.py``, the data is divided into ``nslices`` parts (or slices---think of this like slices on a pizza since the slicing is in the $\varphi$ coordinate). A single slice is then divided into slivers among the available processors. Each processor/MPI task handles a single sliver, and then the slivers are added up together at the end to yield the integral over a single slice. Then, the next slice is set, divided into slivers among the processors, loaded into memory, and analyzed until all slices have been processed. In this sense, the data is "parallel in $\varphi$" rather than "parallel in time." Let's look at the call structure of ``radial_profiles.py``.

As with nearly all scripts in the toolkit, the ``-h`` flag will provide the commands needed to use the script.

#### Aside: Getting MPI4py to work on the Jupyter Nodes

This took a few attempts, so I'll document it here. You should try to execute the following before running the ``radial_profiles.py`` or ``poloidal_slices.py`` scripts.

In [ ]:
from mpi4py import MPI
print(f"Success! MPI version: {MPI.Get_version()}")

If that doesn't work, it means you're missing MPI4py. Getting all of the necessary libraries in the right spot is non-trivial. Here's what I did. After logging into Andes, you want to create a folder in your user directory like this (my user name is "cbambic"):

mkdir /ccs/home/cbambic/jupyter_fallback_libs/

I then needed to update the JSON file for the kernel I was using in my Jupyter notebook. The kernel I'm using is called "Bambic_env" and I see that as the kernel in the upper right hand side of my Jupyter notebook. Whatever kernel name you see/want to use has the JSON file you need to update. Here's my JSON file:

vim /autofs/nccs-svm1_home1/cbambic/.local/share/jupyter/kernels/bambic_env/kernel.json
```
{
 "argv": [
  "/ccs/home/cbambic/.conda/envs/Bambic_env/bin/python",
  "-Xfrozen_modules=off",
  "-m",
  "ipykernel_launcher",
  "-f",
  "{connection_file}"
 ],
 "display_name": "Bambic_env",
 "language": "python",
 "env": {
  "LD_LIBRARY_PATH": "/ccs/home/cbambic/jupyter_fallback_libs:/sw/andes/spack-envs/base/opt/linux-rhel8-x86_64/gcc-9.3.0/openmpi-4.0.4-skxqfeiocc5jtuw3y6dwtnxzzqjp5ffs/lib:/sw/andes/gcc/9.3.0/lib64:/sw/andes/spack-envs/base/opt/linux-rhel8-x86_64/gcc-9.3.0/libfabric-1.8.0-c75evdt25adk7fzoyox4ttc2msi72unv/lib:/usr/lib64:/lib64"
 },
 "metadata": {
  "debugger": true
 }
}
```
Be careful on the formatting here. It makes a major difference. 

Now, I needed to populate my ``jupyter_fallback_libs`` directory with the libraries required by MPI. Here are the list of files in that ``jupyter_fallback_libs`` directory and command I used to copy them over:
- libhcoll.so.1  (cp /opt/mellanox/hcoll/lib/libhcoll.so.1 /ccs/home/cbambic/jupyter_fallback_libs/)
- libibverbs.so.1 (cp /usr/lib64/libibverbs.so.1 /ccs/home/cbambic/jupyter_fallback_libs/)
- libmpi.so.40 (cp /sw/andes/spack-envs/base/opt/linux-rhel8-x86_64/gcc-9.3.0/openmpi-4.0.4-skxqfeiocc5jtuw3y6dwtnxzzqjp5ffs/lib/libmpi.so.40 /ccs/home/cbambic/jupyter_fallback_libs/)
- libocoms.so.0 (cp /opt/mellanox/hcoll/lib/libocoms.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- libpmi2.so.0 (cp /usr/lib64/libpmi2.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- libpmi.so.0 (cp /usr/lib64/libpmi.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- librdmacm.so.1 (cp /usr/lib64/librdmacm.so.1 X)
- libslurm_pmi.so (cp /usr/lib64/slurm/libslurm_pmi.so /ccs/home/cbambic/jupyter_fallback_libs/)
- libucm.so.0 (cp /usr/lib64/libucm.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- libucp.so.0 (cp /usr/lib64/libucp.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- libucs.so.0 (cp /usr/lib64/libucs.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)
- libuct.so.0 (cp /usr/lib64/libuct.so.0 /ccs/home/cbambic/jupyter_fallback_libs/)

You can string much of this into a single command but you need these directories. Now, you should be able to restart the kernel (you may need to go to the Sidebar and on the icon (dark circle with white square) where it says "Running Terminals and Kernels" run "Shut Down All" on kernels and then restart to make sure the JSON that you've updated is being read and the libraries are all pointed in the correct places. After doing this, you should be able to import MPI4py. Try that now.

In [ ]:
from mpi4py import MPI
print(f"Success! MPI version: {MPI.Get_version()}")

I'm hoping this fixes it, but other efforts may be required. This ends the aside.

### Extraction (Again):

Try running the following now that MPI4py is working.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles.py -h

You will see the usage instructions for extracting radial profiles. You can see the same for ``radial_profiles_time.py``:

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles_time.py -h

For ``radial_profiles_time.py`` you see that both a ``start_file`` and ``end_file`` must be specified. Under the hood, the list of files from the start to the end file is divided as evenly as possible among the available processors. Each processor loads in its own dump and writes its own separate output file. In this sense, the script is "parallel in time" since the dumps progress in time.

We now extract radial profiles for our three example simulations, one without radiation transport and two-temperature effects, and one with this extra physics.

Note that the __user__ should specify when to extract quantities related to radiation transport and two-temperature effects. This will also activate CGS units in the extractions.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles.py T65MAD 1500 --shared --title T65MAD --nslices 4 --no-two_temp --no-radiation

Wonderful! Let's do the same thing on the two simulations that include two-temperature and radiation effects. This should only be a short wait so don't go refill your coffee yet.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles.py TRANS_XRB_HIGH_SANE 9000 --shared --title Transition_XRB_HighMdot_SANE_Tutorial --nslices 4 --two_temp --radiation
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles.py TRANS_XRB_HIGH_MAD  9000 --shared --title Transition_XRB_HighMdot_MAD_Tutorial  --nslices 4 --two_temp --radiation

This is great, but we want a couple of files to work with. Let's try to extract at least 20 from each simulation. If you remember from the Exploring H-AMR Data part of the tutorial, we determined the number of dumps for each simulation.

T65MAD: 0 - 1857  
TRANS_XRB_HIGH_SANE: 0 - 12845  
TRANS_XRB_HIGH_MAD: 0 - 11290 

So that we have a couple of files to work with, let's extract about 20 data files from each simulation. This will take a while, so go grab some coffee. We'll use the ``--stride`` parameter for ``radial_profiles_time.py``, which will extract every ``stride``th file.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/radial_profiles_time.py T65MAD 0 1800 --stride 90 --shared --title T65MAD --nslices 4 --no-two_temp --no-radiation

Because of the possibility of node failures or other issues that can interrupt extractions, I run the script ``HAMR_Visualization/Analyze_HAMR_Data/check_if_done.py`` once I've finished performing an extraction. 

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/check_if_done.py -h

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/check_if_done.py T65MAD --profiles

If this runs successfully, then it's time to move on to loading/plotting profiles. If the extraction shows as incomplete, you may need to run the script again using the ``--restart`` flag.

### Loading Radial Profiles

When loading in radial profiles (and poloidal slices), the best strategy is to first determine the valid file numbers. Then, we load in the data corresponding to those file numbers. The ``profiles`` dictionary that's created here has keys corresponding to each of the available file numbers. There is then an additional key for quantities from each file number. For instance, if you wanted to access the mass-accretion rate profile for file number 1500, you would access ``profiles['1500']['Mdot']``.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

Profs = cv.Profiles()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/T65MAD',
    'file_number'  : -1,
    'start_file'   : -1,
    'end_file'     : -1,
    'reduced'      : False
}
file_numbers = Profs.get_fnums(**file_info)
prof_args = {
    'averages_dir' : f'{cv.user_path}/Averages/T65MAD',
    'fnums'        : file_numbers,
    'reduced'      : False
}
profiles = Profs.load_profiles(**prof_args)
profile_fnums = list(profiles.keys())
print(profile_fnums[0], profile_fnums[-1])
print(profiles[f'{profile_fnums[-1]}'].keys())

Note the ``file_number``, ``start_file``, and ``end_file`` inputs. If you wanted to only load in a single value, you would use the ``file_number`` flag and set ``start_file`` and ``end_file`` to "-1." If you wanted to load in data in a range start_file_number to end_file_number, you'd set ``start_file=start_file_number`` and ``end_file=end_file_number``, with ``file_number=-1``. If you just know your start or end file, just set those flags and the code will determine the other values. Please remember to set the parameters that you are not sure of to -1 so that the code knows to infer those.

For comparison, we'll load in a data set with two-temperature and radiation effects.

In [ ]:
Profs = cv.Profiles()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'file_number'  : -1,
    'start_file'   : -1,
    'end_file'     : -1,
    'reduced'      : False
}
file_numbers = Profs.get_fnums(**file_info)
prof_args = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'fnums'        : file_numbers,
    'reduced'      : False
}
profiles = Profs.load_profiles(**prof_args)
profile_fnums = list(profiles.keys())
print(profile_fnums[0], profile_fnums[-1])
print(profiles[f'{profile_fnums[-1]}'].keys())

The quantities shown here are fairly straightforward to interpret; however, because these are radial __profiles__, we had to make some decision as to how to average the data along the $\theta$-direction. In the default radial_profiles scripts, we weight quantities by density. Thus, the following quantities are "density-weighted" and include cuts on sigma ($\sigma$):

- density ('rho')
- ion temperature ('Ti'; when two-temperature effects are not used, this is the gas temperature)
- magnetic pressure ('Pmag')
- thermal pressure ('Pthm')
- volume ('Vol')
- $(\theta - \theta_0)^2$ ('Dtheta2')
- electron temperature ('Te')
- radiation pressure ('Prad')
- ion internal energy ('Eion')
- electron internal energy ('Eelc')
- radial (cylindrical coordinates) 4-velocity ('uR')
- azimuthal (cylindrical coordinates) 4-velocity ('uphi')
- vertical (cylindrical coordinates) 4-velocity ('uz')
- radial (cylindrical coordinates) 3-velocity ('vR')
- azimuthal (cylindrical coordinates) 3-velocity ('vphi')
- vertical (cylindrical coordinates) 3-velocity ('vz')
- radial (cylindrical coordinates) 4-magnetic field ('bR')
- azimuthal (cylindrical coordinates) 4-magnetic field ('bphi')
- vertical (cylindrical coordinates) 4-magnetic field ('bz')

Because they are density-weighted, producing profiles of these quantities requires us to divide out the density-weighted volume. Thus, a density, temperature, or pressure profile (i.e. a quantity that we'll denote as $q$) has a density-weighted average of 

$$ \langle q \rangle_{\rho} = \frac{\int \int q \: \rho \sqrt{-g} dx_2 dx_3}{\int \int \rho \sqrt{-g} dx_2 dx_3} =  \frac{\text{profiles[n][`q']}}{\text{profiles[n][`Vol']}}.$$

_This is a common point of confusion._ Please always remember to check how a quantity was extracted. If your profile seems to be rising sharply (usually with an extra factor of $r^3$), that means that you did not account for dividng out the Jacobian factor/volume element in your calculation. Further, note what quantities have a sigma cut applied. There are times when you can end up with a completely different profile depending on whether a sigma cut (usually 1 or 3---3 is the default) is applied.

The ``load_profiles`` function takes in a number of keyword arguments, the most common of which (and useful of which) is the ``keys`` keyword. Rarely will you want to load in all of the data that's been extracted. By specifying the ``keys``, you can select only the data products that you want. For instance, suppose that you wanted to load in the density (rho), magnetic pressure (Pmag), thermal pressure (Pthm), and radial velocity (vR). You could do this through the call: 

In [ ]:
prof_args = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'fnums'        : file_numbers,
    'reduced'      : False,
    'keys'         : ['rho', 'Pmag', 'Pthm', 'vR', 'Vol', 't', 'r']
}
profiles = Profs.load_profiles(**prof_args)
print(profiles[f'{profile_fnums[-1]}'].keys())

Note that you will usually want to load in the simulation time ('t') and a radius ('r') when you load in data, as time-averaging uses the time key in order to average the data in time. In addition, you usually are trying to plot radial profiles, so you should make sure that you have a radius ('r') available. For now, you could produce a density, magnetic pressure, thermal pressure, and cylindrical radial velocity profile for file number 1500 using:

In [ ]:
# Density
rho = profiles['1500']['rho'] / profiles['1500']['Vol']
# Magnetic pressure
Pmag = profiles['1500']['Pmag'] / profiles['1500']['Vol']
# Thermal pressure
Pmag = profiles['1500']['Pthm'] / profiles['1500']['Vol']
# Radial velocity
vR = profiles['1500']['vR'] / profiles['1500']['Vol']

### Time-averaging Radial Profiles

To time-average profiles after they're read in, use the ``time_avg`` function in the ``Profiles`` class. The optional ``vol`` keyword argument allows you to divide out the volume before time-averaging. Thus, for a density-weighted quantity $q$ with ``vol=False`` (default), we are calculating 

$$ \langle q \rangle_{t} = \frac{1}{t_{\text{end}} - t_{\text{start}}} \int_{t_{\text{start}}}^{t_{\text{end}}} \int \int q \: \rho \sqrt{-g} dx_2 dx_3 dt.$$

However, when ``vol=True``, we are calculating 

$$ \langle q \rangle_{\rho, t} = \frac{1}{t_{\rm end} - t_{\rm start}} \int_{t_{\rm start}}^{t_{\rm end}} \frac{\int \int q \: \rho \sqrt{-g} dx_2 dx_3}{\int \int \rho \sqrt{-g} dx_2 dx_3} dt.$$

In [ ]:
# Divide out the volume
rho  = Profs.time_avg('rho',  profiles, vol=True)
Pmag = Profs.time_avg('Pmag', profiles, vol=True)
Pthm = Profs.time_avg('Pthm', profiles, vol=True)
vR   = Profs.time_avg('vR',   profiles, vol=True)

You can also do the same calculation for a limited number of files.

In [ ]:
# Pick a limited range for files to time-average.
fnums = list(range(9000, 10000))
rho   = Profs.time_avg('rho',  profiles, fnums=fnums, vol=True)

Note that in the toolkit, true time averaging is being done (we are not just adding together dumps and divding by the number of dumps). To turn off the time-weighting, set the ``dt_weight`` parameter to ``False`` in ``Profs.time_avg()``.

### Time-averaging with AMR

Over the course of a H-AMR simulation, AMR might activate and the resolution used for the radial profiles will change. This effect is accounted for in the toolkit. To see this, let's compare a time average taken over a time frame when AMR is inactive vs. AMR is active. In the MAD example simulation, AMR activates at dump 10745.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

Profs = cv.Profiles()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_MAD',
    'file_number'  : -1,
    'start_file'   : -1,
    'end_file'     : -1,
    'reduced'      : False
}
file_numbers = Profs.get_fnums(**file_info)

prof_args = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_MAD',
    'fnums'        : file_numbers,
    'reduced'      : False
}
profiles = Profs.load_profiles(**prof_args)
profile_fnums = list(profiles.keys())
print(profile_fnums[0], profile_fnums[-1])
print(profiles[f'{profile_fnums[-1]}'].keys())

In [ ]:
# Before AMR activates:
print(f"Before AMR activates, the length of the radial profiles is {len(profiles['10744']['r'])}.")
# After AMR activates:
print(f"After AMR activates, the length of the radial profiles is {len(profiles['10745']['r'])}.")

# File numbers not including AMR:
low_res_fnums = list(range(9000, 10000))
rho_low_res   = Profs.time_avg('rho', profiles, fnums=low_res_fnums, vol=True)

print(f"In the time frame without AMR, the length of the radial profiles is {len(rho_low_res)}.")

# File numbers not including AMR:
high_res_fnums = list(range(11000, 11200))
rho_high_res   = Profs.time_avg('rho', profiles, fnums=high_res_fnums, vol=True)

print(f"In the time frame with AMR, the length of the radial profiles is {len(rho_high_res)}.")

# File numbers averaging both refinement levels and upscaling:
mixed_res_fnums = list(range(10000, 11000))
rho_mixed_res   = Profs.time_avg('rho', profiles, fnums=mixed_res_fnums, vol=True)

print(f"Averaging together time frames without AMR (upscaled) and with AMR activated, the length of the radial profiles is {len(rho_mixed_res)}.")

When you execute ``Profs.time_avg()``, under the hood the code is calling a function ``check_AMR()``, which is determining if the resolution changed over the course of this time average. The code then sets a value for ``Profs.fnums0``, which is the first file number associated with the maximum resolution. We can see this here:

In [ ]:
print(f"The frame where AMR first activates is {Profs.fnums0}.")

Consistent with expectations, frame 10745 is the first high-resolution (nr=1120) frame. This is important for plotting as it provides a single radius to plot against the time-averaged profile, i.e., ``profiles[f'{Profs.fnums0}']['r']``.

### Plotting Radial Profiles

Here, we plot 3 density profiles, one for each of our example simulations. We will create three instances of the ``Profiles`` class.

In [ ]:
# This can act as a standalone section.
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

sims     = ['T65MAD', 'Transition_XRB_HighMdot_SANE', 'Transition_XRB_HighMdot_MAD']
Profs    = dict.fromkeys(sims)
profiles = dict.fromkeys(sims)

for sim in sims:
    Profs[sim] = cv.Profiles()

    file_info = {
        'averages_dir' : f'{cv.user_path}/Averages/{sim}',
        'file_number'  : -1,
        'start_file'   : -1,
        'end_file'     : -1,
        'reduced'      : False
    }

    prof_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : Profs[sim].get_fnums(**file_info),
    'reduced'      : False
    }
    
    profiles[sim] = Profs[sim].load_profiles(**prof_args)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import theory
MBH = 10. # 10 Msun black hole
Analytic = theory.Analytic(hr.CONSTANTS, cgs=True)

# Target ranges for time averaging.
fnums = {
    'T65MAD'                       : list(range(1000, 1800)),
    'Transition_XRB_HighMdot_SANE' : list(range(9000, 11000)),
    'Transition_XRB_HighMdot_MAD'  : list(range(9000, 11000))
}

sim_names = {
    'T65MAD'                       : r'$\text{Tilted MAD}$',
    'Transition_XRB_HighMdot_SANE' : r'$\text{SANE with Radiation}$',
    'Transition_XRB_HighMdot_MAD'  : r'$\text{MAD with Radiation}$',
}

colors = {
    'T65MAD'                       : 'black',
    'Transition_XRB_HighMdot_SANE' : 'dodgerblue',
    'Transition_XRB_HighMdot_MAD'  : 'darkorange'  
}

opt = dict(ha='left', va='center', color='black',
           bbox=dict(facecolor='None', edgecolor='none'))

FS = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS
figsize = [10.0, 4.0]

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 2, figure = fig)
ax1  = fig.add_subplot(gs[0, 0])
ax2  = fig.add_subplot(gs[0, 1])

for sim in [sims[0]]:
    rho   = Profs[sim].time_avg('rho', profiles[sim], fnums=fnums[sim], vol=True)
    r     = profiles[sim][f'{Profs[sim].fnums0}']['r']
    ax1.plot(r, rho, color=colors[sim], lw=2, label=sim_names[sim])

for sim in sims[1:]:
    rho   = Profs[sim].time_avg('rho', profiles[sim], fnums=fnums[sim], vol=True)
    r     = profiles[sim][f'{Profs[sim].fnums0}']['r']
    ax2.plot(r, rho, color=colors[sim], lw=2, label=sim_names[sim])

for ax in [ax1, ax2]:
    ax.set_xlabel(r'$r \: (R_g)$', fontsize=FS)
    ax.set_xscale('log')
    ax.set_yscale('log')
    cv.grid(ax, alpha=0.2, color='black')
    ax.legend(loc='lower left', fontsize=FS-8, frameon=False)

ax1.set_ylabel(r'$\rho/\rho_0$', fontsize=FS)
ax2.set_ylabel(r'$\rho \: (\rm{g/cm}^3)$', fontsize=FS)

fig.tight_layout()
plt.show()

Plotting radial profiles is maybe the most frequent task in all of H-AMR analysis, so I've written a script that will plot the most common profiles. Here's an example of how to produce radial profiles for dump 1500 of the tilted MAD, dump 9000 of the SANE with radiation, and dump 9000 of the MAD with radiation.

In [ ]:
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_profiles.py -h

In [ ]:
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_profiles.py T65MAD --file_number 1500 --plot_title Tutorial_T65_MAD_Profiles --r_max 100. --no-two_temp --no-radiation --disable_TeX
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_profiles.py Transition_XRB_HighMdot_SANE --file_number 9000 --plot_title Tutorial_SANE_Radiation_Profiles --r_max 100. --disable_TeX
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_profiles.py Transition_XRB_HighMdot_MAD  --file_number 9000 --plot_title Tutorial_MAD_Radiation_Profiles  --r_max 100. --disable_TeX

Note that I used the ``disable_TeX`` flag rather than asking you to install all of the LaTeX fonts that I prefer to use. On a different occasion, you should install LaTeX so that your plots can take advantage of the improved typesetting.

Your images should be available in the ``PDF`` directory. This is where I like to store plots from my analysis (different than the images for movies). The main thing that you as the user might have to change in these figures is the limits. You'll see at the top of the script that I have a section for those limits:

```
plot_keys = [
    'Mdot',     'rho',        'Temp',     'vR',
    'Edot',  'scaleheight', 'pressures', 'vphi',
    'PhiBH',    'bphi',        'bz',      'vz'
]

plot_info = dict.fromkeys(plot_keys)
for key in plot_keys:
    plot_info[key] = dict.fromkeys(['quant', 'colors', 'axis_id', 'labels', 'title', 'log', 'linthresh'])

user_limits = {
    'Mdot'        : None,
    'rho'         : None,
    'Temp'        : [1.e4,  1.e12] if cgs else [1.e-5, 1.],
    'vR'          : [-1, 0.1],
    'vphi'        : [0.1, 1.1],
    'vz'          : [-0.01, 0.01],
    'Edot'        : None,
    'scaleheight' : [0.01, 2],
    'pressures'   : None,
    'PhiBH'       : [1., 1.e3],
    'bphi'        : None,
    'bz'          : None
} 
```

I believe that this layout should be self-explanatory. 

In [ ]:
from IPython.display import Image, display

display(Image(filename=f'{cv.user_path}/PDF/Tutorial_T65_MAD_Profiles.png'))
display(Image(filename=f'{cv.user_path}/PDF/Tutorial_SANE_Radiation_Profiles.png'))
display(Image(filename=f'{cv.user_path}/PDF/Tutorial_MAD_Radiation_Profiles.png'))

Generally, you can learn a great deal from quickly producing these figures from your runs. They capture the vast majority of the physics going on in the simulations. The only additional quantities that you might want to study immediately are the tilt and precession angles (i.e. angular momentum), which are calculated in the ``Tilt`` class inside of ``coordinates.py``.

### Custom Profiles

Often, you may want to produce a radial profile-like object that is not the output of ``radial_profiles.py`` or ``radial_profiles_time.py``. You should use the ``Custom_Profiles(My_Profiles)`` class and the name of your profile object (in this case, "My_Profiles"). This class operates identically to ``Profiles``, but for the name passed during initialization (e.g. "My_Profiles"). 

### Time Series Data

Radial profiles are the basis for calculating time series data, which forms the basis of any initial exploration of H-AMR data. Producing a time series from profiles is straightforward and is handled by the ``cv.Time_Series()`` class.

In [ ]:
# This is a standalone section
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

sims     = ['T65MAD', 'Transition_XRB_HighMdot_SANE', 'Transition_XRB_HighMdot_MAD']
Profs    = dict.fromkeys(sims)
profiles = dict.fromkeys(sims)

for sim in sims:
    Profs[sim] = cv.Profiles()

    file_info = {
        'averages_dir' : f'{cv.user_path}/Averages/{sim}',
        'file_number'  : -1,
        'start_file'   : -1,
        'end_file'     : -1,
        'reduced'      : False
    }

    prof_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : Profs[sim].get_fnums(**file_info),
    'reduced'      : False
    }
    
    profiles[sim] = Profs[sim].load_profiles(**prof_args)

TS = cv.Time_Series()

time = dict.fromkeys(sims)
Mdot = dict.fromkeys(sims)

for sim in sims:
    time[sim], Mdot[sim] = TS.calc_time_series('Mdot_sigma', profiles[sim], r0=5.) # r0 is the measurement radius in Rg: defaults to the innermost point

Note that the ``r0`` parameter must be passed to ``Time_Series.calc_time_series()`` to ensure that you are measuring the time series at a radius that's beyond the black hole's horizon. Otherwise, the innermost point is used for the time series, and that point is technically within the horizon (i.e., causally disconnected from the global domain).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import theory
MBH = 10. # 10 Msun black hole
Analytic = theory.Analytic(hr.CONSTANTS, cgs=True)
Mdot_Edd = Analytic.calc_Mdot_Edd(MBH=MBH, eta=0.179)

sim_names = {
    'T65MAD'                       : r'$\text{Tilted MAD}$',
    'Transition_XRB_HighMdot_SANE' : r'$\text{SANE with Radiation}$',
    'Transition_XRB_HighMdot_MAD'  : r'$\text{MAD with Radiation}$',
}

colors = {
    'T65MAD'                       : 'black',
    'Transition_XRB_HighMdot_SANE' : 'dodgerblue',
    'Transition_XRB_HighMdot_MAD'  : 'darkorange'
}

FS = 20
plt.rcParams['xtick.labelsize'] = FS
plt.rcParams['ytick.labelsize'] = FS
figsize = [10.0, 4.0]

fig  = plt.figure(figsize = figsize)
gs   = gridspec.GridSpec(1, 1, figure = fig)
ax   = fig.add_subplot(gs[0, 0])

ax.plot(time[sims[0]]/1.e4, Mdot[sims[0]], lw=1, color=colors[sims[0]], label=sim_names[sims[0]])

for sim in sims[1:]:
    ax.plot(time[sim]/1.e4, Mdot[sim]/Mdot_Edd, lw=1, color=colors[sim], label=sim_names[sim])

ax.set_xlabel(r'$\rm{Time} \: (10^4 R_g/c)$', fontsize=FS)
ax.set_ylabel(r'$\dot{M}$', fontsize=FS)
ax.set_yscale('log')
cv.grid(ax, alpha=0.2, color='black')
ax.legend(loc='upper right', fontsize=FS-8, frameon=False)
ax.set_ylim([1.e-1, 1.e4])

fig.tight_layout()
plt.show()

There is a ``HAMR_Visualization/Plot_HAMR_Data/plot_time_series.py`` script that may prove to be very useful. This should create a figure of the mass-accretion rate, horizon-penetrating magnetic flux, and the jet/radiative power.

In [ ]:
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_time_series.py -h

In [ ]:
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_time_series.py T65MAD --r_Mdot 5. --plot_title Tutorial_T65_MAD_Time_Series --no-two_temp --no-radiation --disable_TeX

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'{cv.user_path}/PDF/Tutorial_T65_MAD_Time_Series.png'))

This completes the section on radial profiles and time series data. Please clear the outputs and we'll continue on to studying poloidal slices and stresses, which is the primary basis for calculating energy-outflow rates, angular-momentum transport, and timescales, i.e. the "accretion physics" studied in H-AMR simulations.

## IV) Poloidal Slices and Stresses

### Extraction

Accretion theory is built around mean-field theories: temporally and spatially averaged (i.e. Reynolds-averaged) quantities for a background and fluctuations on top of that background that facilitate energy/angular-momentum transport and heating. The toolkit is designed to facilitate straightforward azimuthal and temporal averaging such that analysis of petabytes of data can be reduced to terabytes. Azimuthal averaging also allows for easy exploration of vertical structure in accretion disks, which allows for direct comparisons to numerical experiments performed in shearing boxes. 

Two main scripts are available for slice extraction:
1. ``HAMR_Visualization/Analyze_HAMR_Data/poloidal_slices.py``
2. ``HAMR_Visualization/Analyze_HAMR_Data/poloidal_slices_time.py``

Without the "_time" convention, the script is designed to operate on a single dump at ultra-high-resolution. With the "_time" convention, the script is designed to operate on many dumps at once. As with nearly all scripts in the toolkit, the ``-h`` flag will provide the commands needed to use the script.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/poloidal_slices.py -h

The main differences between ``poloidal_slices.py`` and ``radial_profiles.py`` are the radiative cooling rates and the opacity calculations when radiation is activated. By toggling off ``cooling``, the script runs dramatically faster (calculating the emssion/absorption opacities and the Coulomb coupling rates is the slowest step). 

Further, the memory demands are more restrictive given that the data products are 2D poloidal slices (i.e. not low-memory 1D radial profiles). You should thus slice your domain more finely, and you should consider using the ``--r_max`` flag to limit the size of the domain averaged over.

Let's execute ``poloidal_slices.py`` on the T65MAD simulation. 

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/poloidal_slices.py T65MAD 1500 --shared --title T65MAD --nslices 8 --no-two_temp --no-radiation --r_max 100.

Here, we obtain enough files to do some interesting time averaging. This may take a bit of time. Either increase the stride or enjoy your coffee break.

In [ ]:
%run ../../HAMR_Visualization/Analyze_HAMR_Data/poloidal_slices_time.py T65MAD 1500 1800 --stride 15 --shared --title T65MAD --nslices 8 --no-two_temp --no-radiation --r_max 100.

We could do the same for the simulations with radiation transport and two-temperature effects.

### Loading Poloidal Slices

Syntactically, handling poloidal slices and radial profiles is almost identical. Thus, we will not repeat most of the details from Part III on Radial Profiles. Instead, we focus on the primary differences with poloidal slices, specifically the calls to ``time_avg``.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

Slices = cv.Slices()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/T65MAD',
    'file_number'  : -1,
    'start_file'   : -1,
    'end_file'     : -1,
    'reduced'      : False
}
file_numbers = Slices.get_fnums(**file_info)
slice_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : file_numbers,
    'reduced'      : False
}
slices = Slices.load_slices(**slice_args)
slice_fnums = list(slices.keys())
print(slice_fnums[0], slice_fnums[-1])
print(slices[f'{slice_fnums[-1]}'].keys())

For comparison, here are the keys for slices from a simulation with two-temperature effects and radiation transport.

In [ ]:
Slices = cv.Slices()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'file_number'  : -1,
    'start_file'   : 9000,
    'end_file'     : 9020,
    'reduced'      : False
}
file_numbers = Slices.get_fnums(**file_info)
slice_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : file_numbers,
    'reduced'      : False
}
slices = Slices.load_slices(**slice_args)
slice_fnums = list(slices.keys())
print(slice_fnums[0], slice_fnums[-1])
print(slices[f'{slice_fnums[-1]}'].keys())

The quantities shown here are fairly straightforward to interpret. These are all azimuthal integrals (i.e. integrals over the $\varphi$-coordinate) that include the volume element $\sqrt{-g} dx_3$ in their definitions.

- density ('rho')
- ion temperature ('Ti'; when two-temperature effects are not used, this is the gas temperature)
- magnetic pressure ('Pmag')
- thermal pressure ('Pthm')
- volume ('Vol')
- enthalpy ('w'; fluid + electromagnetic, and radiation)
- electron temperature ('Te')
- radiation pressure ('Prad')
- ion internal energy ('Eion')
- electron internal energy ('Eelc')
- effective absorption opacity x density ('rhokappa')
- volume with $\sigma > 3$ excluded ('Vol_sigma_cut')
- Coulomb energy exchange rate in the fluid frame x u_t/c ('qie')
- Compton cooling rate in the fluid frame x u_t/c ('qIC')
- Radiative emission rate (excluding Compton cooling) in the fluid frame x u_t/c ('qemi')
- Radiative absorption rate in the fluid_frame x u_t/c ('qabs')
- R-t component of the fluid/hydrodynamic stress-energy tensor ('THD_tR'; used for enthaply fluxes)
- z-t component of the fluid/hydrodynamic stress-energy tensor ('THD_tz'; used for enthalpy fluxes)
- 1-t component of the fluid/hydrodynamic stress-energy tensor ('THD_t1'; used for enthalpy fluxes)
- R-t component of the electromagnetic stress-energy tensor ('TEM_tR'; used for Poynting fluxes)
- z-t component of the electromagnetic stress-energy tensor ('TEM_tz'; used for Poynting fluxes)
- 1-t component of the electromagnetic stress-energy tensor ('TEM_t1'; used for Poynting fluxes)
- Bernoulli parameter ('Be'; not including radiation)
- R-t component of the radiation stress-energy tensor ('R_tR'; used for radiation fluxes)
- z-t component of the radiation stress-energy tensor ('R_tz'; used for radiation fluxes)
- t-t component of the radiation stress-energy tensor ('R_tt'; used for thermal/radiative diffusion timescales)
- 1-t component of the radiation stress-energy tensor ('R_t1'; used for radiation fluxes)
- Bernoulli parameter with radiation ('Be_Rad')
- R-t component of the fluid-frame radiation stress-energy tensor ('R_tR_Fluid'; used for diffusion timescale)
- z-t component of the fluid-frame radiation stress-energy tensor ('R_tz_Fluid'; used for diffusion timescale)
- radial (cylindrical coordinates) 4-velocity ('uR')
- azimuthal (cylindrical coordinates) 4-velocity ('uphi')
- vertical (cylindrical coordinates) 4-velocity ('uz')
- enthalpy-weighted radial (cylindrical coordinates) 4-velocity ('wuR'; used for stresses)
- enthalpy-weighted azimuthal (cylindrical coordinates) 4-velocity ('wuphi'; used for stresses)
- enthalpy-weighted vertical (cylindrical coordinates) 4-velocity ('wuz'; used for stresses)
- enthalpy-weighted (cylindrical coordinates) specific angular momentum ('wu_phi'; used for stresses)
- enthalpy-weighted radial (orthonormal spherical coordinates) 4-velocity ('wur'; used for stresses)
- enthalpy-weighted theta (orthonormal spherical coordinates) 4-velocity ('wuth'; used for stresses)
- radial (cylindrical coordinates) 3-velocity ('vR')
- azimuthal (cylindrical coordinates) 3-velocity ('vphi')
- vertical (cylindrical coordinates) 3-velocity ('vz')
- radial (cylindrical coordinates) 4-magnetic field ('bR')
- azimuthal (cylindrical coordinates) 4-magnetic field ('bphi')
- vertical (cylindrical coordinates) 4-magnetic field ('bz')
- azimuthal (cylindrical coordinates) covariant 4-magnetic field ('b_phi'; used for stresses)
- radial (orthonormal spherical coordinates) 4-magnetic field ('br')
- theta (orthonormal spherical coordinates) 4-magnetic field ('bth')
- density-weighted and sigma-cut radial (cylindrical coordinates) 4-velocity ('rhouR'; used for accretion flowlines)
- density-weighted and sigma-cut azimuthal (cylindrical coordinates) 4-velocity ('rhouphi'; used for accretion flowlines)
- density-weighted and sigma-cut vertical (cylindrical coordinates) 4-velocity ('rhouz'; used for accretion flowlines)
- density-weighted and sigma-cut radial (internal coordinates) 4-velocity ('rhou1'; used for mass-outflow/inflow rates)
- radial (internal coordinates) 3-magnetic field ('B1'; originally used for calculating vector potential contours but this didn't work)

Because the poloidal slices are integrated over azimuth with a weighting from the Jacobian and $dx_3$ (or $d \varphi$), density, temperature, or pressure slices (i.e. a quantity that we'll denote as $q$) have a true average of 

$$ \langle q \rangle_{\varphi} = \frac{\int \int q \: \rho \sqrt{-g} dx_3}{\int \int \sqrt{-g} dx_3} =  \frac{\text{slices[n][`q']}}{\text{slices[n][`Vol']}}.$$

### Time-averaging Poloidal Slices

There are two ways of time-averaging poloidal slices. The first, naive way is to call a similar time-averaging function after loading in all of the slices individually (as we have already done). The syntax/call structure for that works as follows:

In [ ]:
rho_gdet_dx3 = Slices.time_avg('rho', slices)
rho          = Slices.time_avg('rho', slices, vol=True)

However, the preferred way will be to time average as you are loading the data in rather than storing all of the slices in memory. We'll do that now. 

In [ ]:
Slices = cv.Slices()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'file_number'  : -1,
    'start_file'   : 9000,
    'end_file'     : 9020,
    'reduced'      : False
}
file_numbers = Slices.get_fnums(**file_info)
slice_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : file_numbers,
    'reduced'      : False
}
slices = Slices.load_slices_time_averaged(**slice_args)
slice_fnums = list(slices.keys())
print(slice_fnums[0], slice_fnums[-1])
print(slices[f'{slice_fnums[-1]}'].keys())

If I wrote this correctly, it should throw you an error. The reason is that you want to not only supply the ``keys`` for the variables that you would like to read in, you must also provide a dictionary of of ``vols`` with keys of ``keys`` to select if you'd like to divide by the volume element before time-averaging. To just load in the density, ion temperature, and thermal pressure, we would do the following:

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

Slices = cv.Slices()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_SANE',
    'file_number'  : -1,
    'start_file'   : 9000,
    'end_file'     : 9020,
    'reduced'      : False
}
file_numbers = Slices.get_fnums(**file_info)

keys = ['rho', 'Ti', 'Pthm']
vols = dict.fromkeys(keys, True) # divide by volume before time-averaging

slice_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : file_numbers,
    'reduced'      : False,
    'keys'         : keys,
    'vols'         : vols
}
slices = Slices.load_slices_time_averaged(**slice_args)
print(slices.keys())

Note that you receive a number of additional keys beyond the density ('rho'), ion temperature ('Ti'), and thermal pressure ('Pthm'). We include the coordinate arrays ('r', 'theta', and 'phi'), the spin parameter 'a', and the length elements 'dx1', 'dx2', and the time between frames 'dt'. Note that these additional quantities are from one particular file number, usually the first frame at the highest resolution level. If your spin is evolving with time or your output-cadence is changing between frames, you should not rely solely on these quantities.

We can demonstrate AMR by looking at the MAD data set. As a reminder, the resolution jump occurred around frame 10745.

In [ ]:
import sys
import numpy as np
asys_path = '/lustre/orion/ast178/proj-shared/cbambic/HAMR_Visualization/Analyze_HAMR_Data' # you need to specify the path to compute_variables.py in a Jupyter environment
sys.path.insert(0, f'{asys_path}')
import compute_variables as cv
import hamr_read as hr

Slices = cv.Slices()
file_info = {
    'averages_dir' : f'{cv.user_path}/Averages/Transition_XRB_HighMdot_MAD',
    'file_number'  : -1,
    'start_file'   : 10740,
    'end_file'     : 10750,
    'reduced'      : False
}
file_numbers = Slices.get_fnums(**file_info)

keys = ['rho', 'Ti', 'Pthm']
vols = dict.fromkeys(keys, True) # divide by volume before time-averaging

slice_args = {
    'averages_dir' : file_info['averages_dir'],
    'fnums'        : file_numbers,
    'reduced'      : False,
    'keys'         : keys,
    'vols'         : vols
}
slices = Slices.load_slices_time_averaged(**slice_args)
print(slices.keys())

Even though if we were to just load in a single slice from frame 10740, the time-averaged slices are larger because we are upscaling slices 10740-10744 before averaging with 10745-10750.

In [ ]:
single_slice = Slices.load_slices(file_info['averages_dir'], fnums=[10740], keys=['rho', 'Vol'])
rho          = single_slice['10740']['rho'] / single_slice['10740']['Vol']
print(f"The shape of data at frame 10740 is {rho.shape}.")
print(f"The shape of the time-averaged data is {slices['rho'].shape}.")

### Plotting Poloidal Slices

Poloidal slice plots are fundamentally scalar pseudocolor plots, so they're more complex than the lines that we've been working with. I have put together a script for plotting the poloidal slices __for simulations with two-temperature and radiation effects only__; however, it is not as well-developed/tested as some of the others. Here's how to execute it.

In [ ]:
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_slices.py Transition_XRB_HighMdot_SANE --file_number 9000 --plot_title Tutorial_SANE_Radiation_Slices --r_max 40. --plot_vec --disable_TeX
%run ../../HAMR_Visualization/Plot_HAMR_Data/plot_slices.py Transition_XRB_HighMdot_MAD  --file_number 9000 --plot_title Tutorial_MAD_Radiation_Slices  --r_max 40. --plot_vec --disable_TeX

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'{cv.user_path}/PDF/Tutorial_SANE_Radiation_Slices.png'))
display(Image(filename=f'{cv.user_path}/PDF/Tutorial_MAD_Radiation_Slices.png'))

Clear the outputs and we can move on to stresses.

### Extracting Stresses

Because accretion theory is about understanding how the outward transport of angular momentum facilitates the inward transport of mass, measuring the turbulent stresses is often a helpful way to study how angular momentum is transported. We perform extractions of poloidal slices of the turbulent stresses using the ``stresses.py`` and ``stresses_time.py`` scripts.